# Dataset inspection & explainer

This notebook has two jobs, applied to **three separate, directly-comparable
dataset builds** (`configs/dataset/degeneration-dataset-*.yaml` -- same
prompt sample, same rollout budget, differing only in which model produced
the completions):

1. **Explain** how these datasets are built: where each one lives, what
   pipeline stage produced each file, and what every per-token / per-rollout
   / per-prompt column means.
2. **Explore** each dataset as it exists *right now*: verify what's actually
   present vs. still being generated (and give the exact command to produce
   whatever is missing), look at domain/split composition, degeneration-rate
   statistics across all three labeling signals, review LLM-judge calibration
   results for a targeted sample of rollouts, and browse individual
   prompt/rollout generations interactively -- picking which of the three
   datasets to look at with a selector (Section 8).

This notebook is read-only with respect to the datasets themselves, it never
writes into any `output_root`. The only files it writes are exported
prompt/rollout figures under `notebooks/figures/<dataset_tag>/` (Section 8).

## 1. Where these datasets live, and how they're built

### The three datasets

This notebook covers three equally-important, directly-comparable dataset
builds -- same prompt sample (same sources, same `n_prompts`, same seed),
same rollout budget (`n_rollouts_per_prompt=10`, `max_new_tokens=4096`,
`temperature=0.7`, `top_p=0.9`), differing only in which model produced the
completions and where the build is stored:

| tag | model | config |
|---|---|---|
| `apertus-8b-instruct` | `swiss-ai/Apertus-8B-Instruct-2509` (HF Hub) | `configs/dataset/degeneration-dataset-apertus-8b-instruct.yaml` |
| `apertus1p5-capfilter-linear-it8816` | local checkpoint `Apertus-1p5-8B-sft-capfilter-linear-it8816` | `configs/dataset/degeneration-dataset-apertus1p5-capfilter-linear-it8816.yaml` |
| `apertus1p5-sft256k-4200` | local checkpoint `ap1p5-8b-sft-256k-adam-lr6e-5-constant-128n_4200` (tokenizer **BLOCKED**, see that config's header comment) | `configs/dataset/degeneration-dataset-apertus1p5-sft256k-4200.yaml` |

A `tests/test_dataset_gen_paths.py` test
(`test_all_three_datasets_share_identical_sampling_params`) enforces that the
three configs stay identical on every field except `model_name` /
`tokenizer_name` / `output_root` / `work_root`, so any change meant to apply
to all three (e.g. a new domain, see "Adding a new domain" below) must be
made in all three configs, not just one.

### Storage

Every dataset has its own `output_root` (the dataset itself, final, shared,
group-readable) and `work_root` (scratch, resumable intermediates, *not* the
dataset), named after that dataset's tag:

- `output_root`: `/capstor/store/cscs/swissai/infra01/users/mdenegri/degeneration-probe/degeneration-dataset-<tag>/`
- `work_root`: `/capstor/scratch/cscs/mdenegri/degeneration-probe/degeneration-dataset-<tag>_work/`

Group `infra01`; teammates with capstor-store access under that group can
read `output_root` directly. `work_root` holds per-row partial JSON files
written during generation, consolidated into the shards under `output_root`
once a domain finishes -- safe to delete once a domain's shard is fully
consolidated (Section 3 checks this, per dataset).

Both paths are specific to this author's own capstor account -- see
"Building your own copy" below for what to change if *you* run the pipeline
yourself instead of reading one of these shared builds.

### Folder structure under each `output_root`

(identical structure for all three datasets)

```
degeneration-dataset-<tag>/
    manifest.json
    prompts/
        prompts.parquet
    generations/
        <domain>/shard_00000.parquet
    labels/
        <domain>/shard_00000.parquet
    prompt_stats/
        prompt_stats.parquet
    activations/
        <domain>/<prompt_id>/rollout_<k>.safetensors
        manifest.parquet
    llm_judge/
        calibration_sample.parquet
        results_<backend>.parquet
    splits/
        train.jsonl
        val.jsonl
        test_indomain.jsonl
        test_heldout_domains.jsonl
```

What's in each file:

| Path | Contents |
|---|---|
| `manifest.json` | git commit + full config + model config recorded for this build |
| `prompts/prompts.parquet` | every prompt: `prompt_id`, `domain`, `source_dataset`, `source_row_id`, `prompt_text`, `held_out_domain` |
| `generations/<domain>/shard_00000.parquet` | one row per `(prompt_id, rollout_idx)`: `generated_text`, `generated_token_ids`, `per_token_entropy`, `num_tokens`, `stop_reason`, `seed` |
| `labels/<domain>/shard_00000.parquet` | one row per `(prompt_id, rollout_idx)`: `repetition_score`, `entropy` (per-token arrays), plus `lrs_*` -- one exact repeated-substring match per rollout (length/score/positions/gap/period/etc), not a per-token array -- see Section 6 |
| `prompt_stats/prompt_stats.parquet` | one row per `prompt_id`, aggregated across its `n_rollouts_per_prompt` rollouts (Section 5) |
| `activations/<domain>/<prompt_id>/rollout_<k>.safetensors` | cached per-layer hidden states for that rollout |
| `activations/manifest.parquet` | which `(prompt_id, rollout_idx)` have a cached activation |
| `llm_judge/` | LLM-judge calibration labels -- `calibration_sample.parquet` (the fixed sample to judge) + `results_<backend>.parquet` (one resumable manifest per backend) -- see Section 7 |
| `splits/{train,val,test_indomain,test_heldout_domains}.jsonl` | `prompt_id -> split`, one JSON object per line |

`domain` always means one specific *source* (e.g. `deepmath_103k`, `aime_2025`),
never the coarser in-domain/held-out split -- see
`degeneration_probe/dataset_gen/paths.py`'s module docstring. Currently
configured: 5 in-domain sources (`deepmath_103k`, `numinamath_1_5`,
`if_sft_data_verified`, `llama_nemotron`, `aime_2025`) used for
train/val/test_indomain (`aime_2025` split 70/15/15 like the rest, just at
its own much smaller 30-prompt scale -- ~21/4/5), plus 2 sources held out
**entirely** from training for zero-shot testing (`medical_o1`, `codeforces`)
that only ever appear in `test_heldout_domains`. `codeforces` additionally
restricts its source pool
to `rating >= 2000` via a source's optional `filter_field`/`min_value` keys
(applied before the same random sample every other source uses -- see
"Adding a new domain" below), rather than sampling uniformly over the whole
`open-r1/codeforces` dataset like the other sources do over their own.

### Pipeline stages, in order (each depends on the previous stage's output)

1. **`build_prompts.py`**: samples prompts from each configured HF source,
   assigns `prompt_id` / `domain` / `held_out_domain`, writes
   `prompts.parquet` + the 4 split files. CPU-only, run directly.
2. **`generate.py`**: for each domain, runs `n_rollouts_per_prompt` (10)
   rollouts x every prompt through the model (`max_new_tokens=4096`,
   `temperature=0.7`, `top_p=0.9`), writes `generations/<domain>/shard_00000.parquet`.
   **Needs a GPU**: runs as a SLURM job via `cluster/utils/dataset/generate.sbatch`,
   one independent job chain per domain (resumable, a chain that hits the
   12h wall-clock limit just picks up where it left off in its next wave).
3. **`label.py`**: for each already-generated domain, computes the
   per-token repetition/entropy signals plus the whole-rollout LRS match,
   and aggregates them into `prompt_stats.parquet`. CPU-only, run directly,
   takes seconds per domain. By design, it overwrites `prompt_stats.parquet`
   wholesale with exactly the domains passed via `--domains` on that
   invocation, rather than merging with a previous run -- so always pass
   every currently-fully-generated domain together, not just the
   newly-finished ones. Section 3 accounts for this when it prints the
   recreate command.
4. **`cache_activations.py`**: re-runs a teacher-forced forward pass per
   rollout to cache every layer's hidden states. Needs a GPU; all domains run
   sequentially in one job (not split per-domain like `generate.py`) because
   `activations/manifest.parquet` is a single file shared across domains, and
   concurrent per-domain jobs would race on writing it. Fully complete for
   the current build (Section 3 confirms this).
5. **`llm_judge.py`**: selects a fixed calibration sample (two mutually
   exclusive strata: rollouts that hit the token cap, and rollouts
   heuristically flagged as degenerating that nonetheless reached EOS on
   their own), then judges each one with an LLM that sees only the prompt +
   completion text (blind to the heuristic scores). Three interchangeable
   backends (`--backend anthropic|claude_agent_sdk|openrouter`), swappable
   with a CLI flag; results are a resumable, per-backend manifest. CPU-only
   driver (the judging itself happens over the network/subprocess, not local
   compute), run directly. See Section 7.

### Building your own copy

To generate your own build rather than reading one of the three shared ones
above, copy whichever of the three configs is closest to what you want and
override `output_root` / `work_root` -- the defaults there (and
`DatasetGenConfig`'s own field defaults) point at this author's own capstor
paths, which you most likely can't write to:

```yaml
output_root: "/capstor/store/cscs/swissai/infra01/users/$USER/degeneration-probe/degeneration-dataset-<tag>"
work_root: "/capstor/scratch/cscs/$USER/degeneration-probe/degeneration-dataset-<tag>_work"
```

Then run the 5 pipeline stages above in order against your copy of the
config, starting with `build_prompts.py`. See "Adding a new domain" below if
you also want to change which HF sources are included, not just where the
output goes.

### Adding a new domain

A "domain" is just one more entry in a config's `in_domain_sources` or
`held_out_sources` list. Since all three datasets are meant to share the same
prompt sample, a new domain should be added to **all three** configs
(`configs/dataset/degeneration-dataset-*.yaml`) with identical `name` /
`hf_repo` / `hf_subset` / `hf_split` / `prompt_field` / `n_prompts`, not just
one -- otherwise `test_all_three_datasets_share_identical_sampling_params`
will fail. To add one (say you found a new HF dataset):

1. **Pick a `name`** -- a short, filesystem-safe identifier (e.g.
   `deepmath_103k`). This becomes the subfolder name under `generations/`,
   `labels/`, `activations/`, and the `prompt_id` prefix
   (`f"{name}_{index:05d}"`), so it must be unique across both source lists.
2. **Add a source dict** with all of `SOURCE_REQUIRED_KEYS`
   (`degeneration_probe/dataset_gen/config.py`): `name`, `hf_repo`,
   `hf_subset` (or `null`), `hf_split`, `prompt_field`, `n_prompts`. Two
   further keys, `filter_field` / `min_value`, are optional (unset by
   default) and restrict the pool sampled from to rows where
   `row[filter_field] >= min_value`, applied before the same random sample
   every source goes through -- `codeforces` uses this to sample only from
   problems rated 2000+ (Codeforces Master level or harder).
3. **Find the right `prompt_field`** -- the HF column holding the prompt.
   You choose the exact column name; `build_prompts.py` accepts either a flat
   string column (e.g. DeepMath's `"question"`) or a conversation-list column,
   a list of `{"role": ..., "content": ...}` dicts (e.g.
   `if_sft_data_verified`'s `"messages"`, `llama_nemotron`'s `"input"`) --
   `extract_prompt_text` auto-detects which and pulls the first user turn's
   text out of the latter. Check the dataset's actual schema on the HF Hub
   (not just its card) before picking this: `if_sft_data_verified`'s card
   implies a `"prompt"` column that doesn't actually exist -- the real
   column is `"messages"`.
4. **Choose in-domain vs. held-out.** Default to `held_out_sources` for a
   brand-new source -- it's reserved entirely for out-of-distribution eval
   (never trained on, only ever appears in `test_heldout_domains`), which is
   the safe choice before you know how in-distribution it actually looks.
   Move it to `in_domain_sources` instead if you specifically want it split
   across train/val/test_indomain like the 4 existing in-domain sources --
   there's no per-source override of `split_fractions`, every in-domain
   source is split with the same fractions.
5. **Large HF repos are handled automatically.** If the repo's total file
   size exceeds `LARGE_REPO_STREAMING_THRESHOLD_BYTES` (5 GiB),
   `build_prompts.py` switches to streaming mode on its own and only scans
   the first `STREAMING_PREFIX_SCAN_SIZE` (20,000) rows before sampling --
   no action needed, but be aware the sample is drawn from the front of the
   stream rather than uniformly over the whole split (see `build_prompts.py`'s
   module docstring for that tradeoff).
6. **Rebuild the prompt pool and splits, for each of the three configs.**
   This is the only stage that reads the new source, and it reassigns splits
   for every domain from scratch, so it must be rerun even though only one
   source changed:
   ```
   .venv/bin/python -m degeneration_probe.dataset_gen.build_prompts --config configs/dataset/degeneration-dataset-apertus-8b-instruct.yaml
   .venv/bin/python -m degeneration_probe.dataset_gen.build_prompts --config configs/dataset/degeneration-dataset-apertus1p5-capfilter-linear-it8816.yaml
   .venv/bin/python -m degeneration_probe.dataset_gen.build_prompts --config configs/dataset/degeneration-dataset-apertus1p5-sft256k-4200.yaml
   ```
7. **Generate, label, cache activations, and judge, for each dataset.** All
   four stages are resumable, so this only computes what's new (shown here
   for one dataset's config -- repeat with each of the three):
   ```
   sbatch --export=ALL,DOMAIN=<new_domain>,CONFIG=configs/dataset/degeneration-dataset-<tag>.yaml cluster/utils/dataset/generate.sbatch

   .venv/bin/python -m degeneration_probe.dataset_gen.label \
       --config configs/dataset/degeneration-dataset-<tag>.yaml \
       --domains <every fully-generated domain, old + new>   # see the label.py note above

   sbatch --export=ALL,CONFIG=configs/dataset/degeneration-dataset-<tag>.yaml cluster/utils/dataset/cache_activations.sbatch
       # always processes every domain, but skips every already-cached rollout --
       # effectively a new-domain-only run in practice

   .venv/bin/python -m degeneration_probe.dataset_gen.llm_judge \
       --config configs/dataset/degeneration-dataset-<tag>.yaml --backend claude_agent_sdk
   ```
   `label.py`'s `--domains` list must include every already-labeled domain
   too, not just the new one -- `prompt_stats.parquet` is overwritten
   wholesale with exactly what's passed (see the `label.py` step above and
   Section 3, which prints this exact command for you once generation is
   complete).

### Dependencies to run any of this yourself

- Repo checked out at `/iopsstor/scratch/cscs/$USER/degeneration-probe`, at
  the exact commit recorded in each dataset's `manifest.json`'s
  `pipeline_git_commit` (Section 2 reads and prints it, per dataset).
- **CPU-only steps** (`build_prompts.py`, `label.py`, this notebook): the
  repo's `uv`-managed virtualenv at `.venv/`, run `uv sync` from the repo
  root if it doesn't exist yet. This notebook additionally needs `ipywidgets`
  for Section 8's interactive cell (already pinned in `pyproject.toml`).
- **GPU steps** (`generate.py`, `cache_activations.py`): submitted through
  SLURM using the container image pinned in `cluster/env.toml`, plus a valid
  HF token at `~/keys/.hf_token` (needed for gated/rate-limited HF datasets
  and the model itself).

## 2. Setup

In [1]:
import json
import textwrap
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from degeneration_probe.dataset_gen import paths
from degeneration_probe.dataset_gen import label as label_module
from degeneration_probe.dataset_gen import llm_judge as llm_judge_module
from degeneration_probe.dataset_gen.config import DatasetGenConfig
from degeneration_probe.dataset_gen.manifest import read_manifest

pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# The three equally-important, directly-comparable dataset builds (Section 1) --
# every section below loops over all three and shows one table/plot per dataset,
# keyed by this short tag. Point this at your own config(s) to inspect a
# different build (see Section 1's "Building your own copy").
DATASET_CONFIG_PATHS = {
    "apertus-8b-instruct": REPO_ROOT / "configs" / "dataset" / "degeneration-dataset-apertus-8b-instruct.yaml",
    "apertus1p5-capfilter-linear-it8816": REPO_ROOT / "configs" / "dataset" / "degeneration-dataset-apertus1p5-capfilter-linear-it8816.yaml",
    "apertus1p5-sft256k-4200": REPO_ROOT / "configs" / "dataset" / "degeneration-dataset-apertus1p5-sft256k-4200.yaml",
}
DATASET_TAGS = list(DATASET_CONFIG_PATHS)
SPLIT_NAMES = ["train", "val", "test_indomain", "test_heldout_domains"]

# `ds[tag]` accumulates everything about that one dataset as the notebook loads
# more of it, cell by cell -- config/paths here, dataframes in Section 4, derived
# tables in Section 6/7. Keeps every later cell a loop over the same dict instead
# of one set of bare globals per dataset.
ds = {}
for tag, config_path in DATASET_CONFIG_PATHS.items():
    config = DatasetGenConfig.from_yaml(config_path)
    ds[tag] = {
        "config_path": config_path,
        "config": config,
        "DOMAINS": sorted(paths.configured_domain_names(config)),
    }

for tag, d in ds.items():
    print(f"[{tag}]")
    print(f"  config           : {d['config_path'].relative_to(REPO_ROOT)}")
    print(f"  output_root      : {d['config'].output_root}")
    print(f"  Configured domains ({len(d['DOMAINS'])}): {d['DOMAINS']}")

W0720 19:25:59.805000 126820 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


W0720 19:25:59.943000 126820 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


[apertus-8b-instruct]
  config           : configs/dataset/degeneration-dataset-apertus-8b-instruct.yaml
  output_root      : /capstor/store/cscs/swissai/infra01/users/mdenegri/degeneration-probe/degeneration-dataset-apertus-8b-instruct
  Configured domains (7): ['aime_2025', 'codeforces', 'deepmath_103k', 'if_sft_data_verified', 'llama_nemotron', 'medical_o1', 'numinamath_1_5']
[apertus1p5-capfilter-linear-it8816]
  config           : configs/dataset/degeneration-dataset-apertus1p5-capfilter-linear-it8816.yaml
  output_root      : /capstor/store/cscs/swissai/infra01/users/mdenegri/degeneration-probe/degeneration-dataset-apertus1p5-capfilter-linear-it8816
  Configured domains (7): ['aime_2025', 'codeforces', 'deepmath_103k', 'if_sft_data_verified', 'llama_nemotron', 'medical_o1', 'numinamath_1_5']
[apertus1p5-sft256k-4200]
  config           : configs/dataset/degeneration-dataset-apertus1p5-sft256k-4200.yaml
  output_root      : /capstor/store/cscs/swissai/infra01/users/mdenegri/degene

In [2]:
import os

from transformers import AutoTokenizer

# The tokenizer is needed both to locate onset_quote inside a rollout's token
# sequence (Section 7's onset-quote validation) and to decode individual
# generated_token_ids for Section 8's per-token visualization -- everything
# else in this notebook works off the already-decoded `generated_text` column.
# Point HF_HOME at the same cache generate.py filled in (a different
# filesystem than REPO_ROOT), and load offline since the login node has no
# guaranteed internet access.
HF_CACHE_DIR = Path(f"/capstor/scratch/cscs/{os.environ['USER']}/degeneration-probe/hf_cache")
os.environ.setdefault("HF_HOME", str(HF_CACHE_DIR))
os.environ.setdefault("HF_HUB_CACHE", str(HF_CACHE_DIR))
os.environ.setdefault("HF_HUB_OFFLINE", "1")

# Each dataset can point at its own model/tokenizer (two of the three use a local
# checkpoint), so the tokenizer is loaded lazily, per dataset tag, and cached here
# rather than eagerly for all three -- mirrors generate.py/cache_activations.py's
# own `tokenizer_name or model_name` fallback (see the apertus1p5-sft256k-4200
# config's header comment for why that fallback is currently BLOCKED for that
# one dataset specifically -- this notebook doesn't second-guess it, just mirrors
# the same config-driven fallback the pipeline itself uses).
_tokenizer_cache = {}


def get_tokenizer(tag):
    if tag not in _tokenizer_cache:
        config = ds[tag]["config"]
        tokenizer_name = config.tokenizer_name or config.model_name
        _tokenizer_cache[tag] = AutoTokenizer.from_pretrained(tokenizer_name, local_files_only=True)
    return _tokenizer_cache[tag]


In [3]:
for tag, d in ds.items():
    config = d["config"]
    manifest_file = paths.manifest_path(config)
    print(f"[{tag}] manifest.json", end=": ")
    if manifest_file.exists():
        manifest = read_manifest(manifest_file)
        print("OK")
        print(f"  pipeline_git_commit : {manifest['pipeline_git_commit']}")
        print(f"  created_at          : {manifest['created_at']}")
        print(f"  model_config        : {manifest['model_config']}")
    else:
        cfg_rel = d["config_path"].relative_to(REPO_ROOT)
        print("MISSING. To write it, run from the repo root (.venv, CPU-only):")
        print()
        print("  .venv/bin/python -c \"")
        print("from degeneration_probe.dataset_gen.config import DatasetGenConfig")
        print("from degeneration_probe.dataset_gen.manifest import write_manifest")
        print(f"config = DatasetGenConfig.from_yaml('{cfg_rel}')")
        print("write_manifest(config)\"")
    print()

[apertus-8b-instruct] manifest.json: OK
  pipeline_git_commit : 6dd250aaf7469f00cfdcec645d93c36ae11bea92
  created_at          : 2026-07-10T20:04:55.654722+00:00
  model_config        : {'hidden_size': 4096, 'num_hidden_layers': 32, 'vocab_size': 131072, 'model_type': 'apertus'}

[apertus1p5-capfilter-linear-it8816] manifest.json: MISSING. To write it, run from the repo root (.venv, CPU-only):

  .venv/bin/python -c "
from degeneration_probe.dataset_gen.config import DatasetGenConfig
from degeneration_probe.dataset_gen.manifest import write_manifest
config = DatasetGenConfig.from_yaml('configs/dataset/degeneration-dataset-apertus1p5-capfilter-linear-it8816.yaml')
write_manifest(config)"

[apertus1p5-sft256k-4200] manifest.json: MISSING. To write it, run from the repo root (.venv, CPU-only):

  .venv/bin/python -c "
from degeneration_probe.dataset_gen.config import DatasetGenConfig
from degeneration_probe.dataset_gen.manifest import write_manifest
config = DatasetGenConfig.from_yaml('co

## 3. Completeness check

Verifies every stage's output actually exists and is fully populated, and
prints the exact command to (re)build anything that's missing or partial.
Nothing in this section modifies the dataset.

In [4]:
def _expected_prompt_count(config, domain):
    for source in [*config.in_domain_sources, *config.held_out_sources]:
        if source["name"] == domain:
            return source["n_prompts"]
    raise KeyError(domain)


for tag, d in ds.items():
    config = d["config"]
    cfg_rel = d["config_path"].relative_to(REPO_ROOT)
    prompts_ok = paths.prompts_path(config).exists()
    splits_ok = all(paths.split_path(config, name).exists() for name in SPLIT_NAMES)
    d["prompts_ok"] = prompts_ok
    d["splits_ok"] = splits_ok

    print(f"[{tag}]")
    print(f"  [{'OK' if prompts_ok else 'MISSING'}] prompts.parquet")
    print(f"  [{'OK' if splits_ok else 'MISSING'}] splits/*.jsonl")
    if not (prompts_ok and splits_ok):
        print()
        print("  To (re)build prompts + splits, run from the repo root (.venv, CPU-only):")
        print(f"    .venv/bin/python -m degeneration_probe.dataset_gen.build_prompts --config {cfg_rel}")
    print()

[apertus-8b-instruct]
  [OK] prompts.parquet
  [OK] splits/*.jsonl

[apertus1p5-capfilter-linear-it8816]
  [OK] prompts.parquet
  [OK] splits/*.jsonl

[apertus1p5-sft256k-4200]
  [OK] prompts.parquet
  [OK] splits/*.jsonl



In [5]:
for tag, d in ds.items():
    config = d["config"]
    n_rollouts = config.n_rollouts_per_prompt

    actual_prompt_counts = (
        pd.read_parquet(paths.prompts_path(config))["domain"].value_counts().to_dict()
        if d["prompts_ok"] else {}
    )

    rows = []
    for domain in d["DOMAINS"]:
        expected_prompts = _expected_prompt_count(config, domain)
        actual_prompts = actual_prompt_counts.get(domain, 0)
        expected_rows = actual_prompts * n_rollouts

        gen_path = paths.generations_shard_path(config, domain, 0)
        gen_rows = len(pd.read_parquet(gen_path)) if gen_path.exists() else 0
        gen_complete = expected_rows > 0 and gen_rows >= expected_rows

        lbl_path = paths.labels_shard_path(config, domain, 0)
        lbl_rows = len(pd.read_parquet(lbl_path)) if lbl_path.exists() else 0
        lbl_complete = gen_rows > 0 and lbl_rows >= gen_rows

        rows.append({
            "domain": domain,
            "prompts": f"{actual_prompts}/{expected_prompts}",
            "generations": f"{gen_rows}/{expected_rows}" if expected_rows else "n/a",
            "gen_complete": gen_complete,
            "labels": f"{lbl_rows}/{gen_rows}" if gen_rows else "n/a",
            "lbl_complete": lbl_complete,
        })

    d["rows"] = rows
    d["completeness_df"] = pd.DataFrame(rows).set_index("domain")
    print(f"[{tag}]")
    display(d["completeness_df"])

[apertus-8b-instruct]


,prompts,generations,gen_complete,labels,lbl_complete
domain,,,,,
aime_2025,30/30,300/300,True,300/300,True
codeforces,600/600,6000/6000,True,6000/6000,True
deepmath_103k,600/600,6000/6000,True,6000/6000,True
if_sft_data_verified,600/600,6000/6000,True,6000/6000,True
llama_nemotron,600/600,6000/6000,True,6000/6000,True
medical_o1,600/600,6000/6000,True,6000/6000,True
numinamath_1_5,600/600,6000/6000,True,6000/6000,True


[apertus1p5-capfilter-linear-it8816]


,prompts,generations,gen_complete,labels,lbl_complete
domain,,,,,
aime_2025,30/30,300/300,True,300/300,True
codeforces,600/600,6000/6000,True,6000/6000,True
deepmath_103k,600/600,6000/6000,True,6000/6000,True
if_sft_data_verified,600/600,6000/6000,True,6000/6000,True
llama_nemotron,600/600,6000/6000,True,6000/6000,True
medical_o1,600/600,6000/6000,True,6000/6000,True
numinamath_1_5,600/600,6000/6000,True,6000/6000,True


[apertus1p5-sft256k-4200]


,prompts,generations,gen_complete,labels,lbl_complete
domain,,,,,
aime_2025,30/30,300/300,True,300/300,True
codeforces,600/600,6000/6000,True,6000/6000,True
deepmath_103k,600/600,6000/6000,True,6000/6000,True
if_sft_data_verified,600/600,6000/6000,True,6000/6000,True
llama_nemotron,600/600,6000/6000,True,6000/6000,True
medical_o1,600/600,6000/6000,True,6000/6000,True
numinamath_1_5,600/600,6000/6000,True,6000/6000,True


In [6]:
for tag, d in ds.items():
    cfg_rel = d["config_path"].relative_to(REPO_ROOT)
    rows = d["rows"]
    gen_incomplete = [r["domain"] for r in rows if not r["gen_complete"]]
    gen_complete_domains = [r["domain"] for r in rows if r["gen_complete"]]
    lbl_stale = [r["domain"] for r in rows if r["gen_complete"] and not r["lbl_complete"]]

    print(f"[{tag}]")
    if gen_incomplete:
        print("  Generation incomplete/not started for:", gen_incomplete)
        print("  Resume/start each with (resumable -- safe to rerun, GPU job):")
        for domain in gen_incomplete:
            print(f"    sbatch --export=ALL,DOMAIN={domain},CONFIG={cfg_rel} cluster/utils/dataset/generate.sbatch")
        print()

    if lbl_stale:
        domains_arg = " ".join(gen_complete_domains)
        print("  Fully-generated domains needing (re)labeling:", lbl_stale)
        print("  NOTE: label.py overwrites prompt_stats.parquet with exactly the --domains passed --")
        print("  always pass every currently fully-generated domain together, not just the new ones:")
        print()
        print(f"    .venv/bin/python -m degeneration_probe.dataset_gen.label --config {cfg_rel} --domains {domains_arg}")
        print()

    if not gen_incomplete and not lbl_stale:
        print("  All configured domains: generation + labels complete.")
    print()

[apertus-8b-instruct]
  All configured domains: generation + labels complete.

[apertus1p5-capfilter-linear-it8816]
  All configured domains: generation + labels complete.

[apertus1p5-sft256k-4200]
  All configured domains: generation + labels complete.



In [7]:
for tag, d in ds.items():
    config = d["config"]
    cfg_rel = d["config_path"].relative_to(REPO_ROOT)
    print(f"[{tag}]")

    prompt_stats_ok = paths.prompt_stats_path(config).exists()
    print(f"  [{'OK' if prompt_stats_ok else 'MISSING'}] prompt_stats.parquet", end="")
    if prompt_stats_ok:
        prompt_stats_df = pd.read_parquet(paths.prompt_stats_path(config))
        ps_domains = sorted(prompt_stats_df["domain"].unique())
        print(f"  (covers domains: {ps_domains})")
    else:
        prompt_stats_df = pd.DataFrame()
        print()
    d["prompt_stats_df"] = prompt_stats_df

    act_manifest_path = paths.activations_manifest_path(config)
    n_act = len(pd.read_parquet(act_manifest_path)) if act_manifest_path.exists() else 0
    # Expected total = sum of n_rollouts across all prompts (one activation file per rollout).
    total_expected_rollouts = int(prompt_stats_df["n_rollouts"].sum()) if prompt_stats_ok else 0
    act_complete = prompt_stats_ok and n_act >= total_expected_rollouts and total_expected_rollouts > 0
    print(
        f"  activations/manifest.parquet: {n_act}/{total_expected_rollouts} cached rollout(s)"
        f"  [{'complete' if act_complete else 'in progress'}]"
    )

    llm_judge_sample_ok = paths.llm_judge_sample_path(config).exists()
    print(f"  [{'OK' if llm_judge_sample_ok else 'MISSING'}] llm_judge/calibration_sample.parquet", end="")
    if llm_judge_sample_ok:
        _llm_judge_sample_peek = pd.read_parquet(paths.llm_judge_sample_path(config))
        print(f"  ({len(_llm_judge_sample_peek)} rollouts: {_llm_judge_sample_peek['stratum'].value_counts().to_dict()})")
    else:
        print()
        print("    To build it (and start/resume judging), submit the judging job (CPU-only driver --")
        print("    needs generations + labels already present for every domain; rotates across multiple")
        print("    accounts' OAuth tokens automatically, see Section 7):")
        print(f"      sbatch --export=ALL,CONFIG={cfg_rel} cluster/utils/dataset/judge.sbatch")

    # One resumable manifest per backend -- glob rather than a fixed list so a new backend's
    # results show up here without a notebook edit.
    llm_judge_result_files = (
        sorted(paths.llm_judge_dir(config).glob("results_*.parquet"))
        if paths.llm_judge_dir(config).exists() else []
    )
    d["llm_judge_result_files"] = llm_judge_result_files
    if llm_judge_result_files:
        for f in llm_judge_result_files:
            backend_name = f.stem.removeprefix("results_")
            _status_counts = pd.read_parquet(f)["status"].value_counts().to_dict()
            print(f"  [OK] llm_judge/results_{backend_name}.parquet: {_status_counts}")
    else:
        print("  [MISSING] llm_judge/results_*.parquet -- no backend has been run yet")
    print()

[apertus-8b-instruct]
  [OK] prompt_stats.parquet  (covers domains: ['aime_2025', 'codeforces', 'deepmath_103k', 'if_sft_data_verified', 'llama_nemotron', 'medical_o1', 'numinamath_1_5'])
  activations/manifest.parquet: 36300/36300 cached rollout(s)  [complete]
  [OK] llm_judge/calibration_sample.parquet  (890 rollouts: {'truncated': 890})


  [OK] llm_judge/results_anthropic.parquet: {'failed': 890}


  [OK] llm_judge/results_claude_agent_sdk.parquet: {'ok': 1170, 'failed': 74}

[apertus1p5-capfilter-linear-it8816]
  [OK] prompt_stats.parquet  (covers domains: ['aime_2025', 'codeforces', 'deepmath_103k', 'if_sft_data_verified', 'llama_nemotron', 'medical_o1', 'numinamath_1_5'])
  activations/manifest.parquet: 0/36300 cached rollout(s)  [in progress]
  [OK] llm_judge/calibration_sample.parquet  (1348 rollouts: {'truncated': 1348})
  [OK] llm_judge/results_claude_agent_sdk.parquet: {'ok': 2837, 'failed': 400}

[apertus1p5-sft256k-4200]
  [OK] prompt_stats.parquet  (covers domains: ['aime_2025', 'codeforces', 'deepmath_103k', 'if_sft_data_verified', 'llama_nemotron', 'medical_o1', 'numinamath_1_5'])
  activations/manifest.parquet: 0/36300 cached rollout(s)  [in progress]
  [OK] llm_judge/calibration_sample.parquet

  (1193 rollouts: {'truncated': 1193})


  [OK] llm_judge/results_claude_agent_sdk.parquet: {'ok': 1111, 'failed': 82}



## 4. Load available tables + data-integrity checks

Only loads domains whose generations/labels actually exist on disk right now
(per Section 3) -- a still-generating domain is skipped with a note rather
than raising.

In [8]:
for tag, d in ds.items():
    config = d["config"]
    DOMAINS = d["DOMAINS"]

    available_gen_domains = [dm for dm in DOMAINS if paths.generations_shard_path(config, dm, 0).exists()]
    available_lbl_domains = [dm for dm in DOMAINS if paths.labels_shard_path(config, dm, 0).exists()]
    d["available_gen_domains"] = available_gen_domains
    d["available_lbl_domains"] = available_lbl_domains

    prompts_df = pd.read_parquet(paths.prompts_path(config)) if d["prompts_ok"] else pd.DataFrame()

    gen_frames = []
    for domain in available_gen_domains:
        df = pd.read_parquet(paths.generations_shard_path(config, domain, 0))
        df["domain"] = domain
        gen_frames.append(df)
    generations_df = pd.concat(gen_frames, ignore_index=True) if gen_frames else pd.DataFrame()

    label_frames = []
    for domain in available_lbl_domains:
        df = pd.read_parquet(paths.labels_shard_path(config, domain, 0))
        df["domain"] = domain
        label_frames.append(df)
    labels_df = pd.concat(label_frames, ignore_index=True) if label_frames else pd.DataFrame()

    splits = {}
    for name in SPLIT_NAMES:
        split_file = paths.split_path(config, name)
        splits[name] = (
            [json.loads(line)["prompt_id"] for line in open(split_file) if line.strip()]
            if split_file.exists() else []
        )

    # LLM-judge calibration data (Section 7): the fixed sample to judge, plus every backend's
    # resumable results manifest, tagged with a "backend" column so they can be told apart once
    # concatenated. `llm_judge_result_files` was already globbed in Section 3.
    llm_judge_sample_df = (
        pd.read_parquet(paths.llm_judge_sample_path(config))
        if paths.llm_judge_sample_path(config).exists() else pd.DataFrame()
    )
    judge_frames = []
    for f in d["llm_judge_result_files"]:
        df = pd.read_parquet(f)
        df["backend"] = f.stem.removeprefix("results_")
        judge_frames.append(df)
    llm_judge_results_df = pd.concat(judge_frames, ignore_index=True) if judge_frames else pd.DataFrame()

    d["prompts_df"] = prompts_df
    d["generations_df"] = generations_df
    d["labels_df"] = labels_df
    d["splits"] = splits
    d["llm_judge_sample_df"] = llm_judge_sample_df
    d["llm_judge_results_df"] = llm_judge_results_df

    print(f"[{tag}]")
    print(f"  prompts_df       : {len(prompts_df)} rows")
    print(f"  generations_df   : {len(generations_df)} rows  (domains: {available_gen_domains})")
    print(f"  labels_df        : {len(labels_df)} rows  (domains: {available_lbl_domains})")
    print(f"  prompt_stats_df  : {len(d['prompt_stats_df'])} rows")
    for name in SPLIT_NAMES:
        print(f"  splits[{name!r}]".ljust(30) + f": {len(splits[name])}")
    print(f"  llm_judge_sample_df  : {len(llm_judge_sample_df)} rows")
    print(
        f"  llm_judge_results_df : {len(llm_judge_results_df)} rows"
        + (f"  (backends: {sorted(llm_judge_results_df['backend'].unique())})" if not llm_judge_results_df.empty else "")
    )
    print()

[apertus-8b-instruct]
  prompts_df       : 3630 rows
  generations_df   : 36300 rows  (domains: ['aime_2025', 'codeforces', 'deepmath_103k', 'if_sft_data_verified', 'llama_nemotron', 'medical_o1', 'numinamath_1_5'])
  labels_df        : 36300 rows  (domains: ['aime_2025', 'codeforces', 'deepmath_103k', 'if_sft_data_verified', 'llama_nemotron', 'medical_o1', 'numinamath_1_5'])
  prompt_stats_df  : 3630 rows
  splits['train']             : 1701
  splits['val']               : 364
  splits['test_indomain']     : 365
  splits['test_heldout_domains']: 1200
  llm_judge_sample_df  : 890 rows
  llm_judge_results_df : 2134 rows  (backends: ['anthropic', 'claude_agent_sdk'])



[apertus1p5-capfilter-linear-it8816]
  prompts_df       : 3630 rows
  generations_df   : 36300 rows  (domains: ['aime_2025', 'codeforces', 'deepmath_103k', 'if_sft_data_verified', 'llama_nemotron', 'medical_o1', 'numinamath_1_5'])
  labels_df        : 36300 rows  (domains: ['aime_2025', 'codeforces', 'deepmath_103k', 'if_sft_data_verified', 'llama_nemotron', 'medical_o1', 'numinamath_1_5'])
  prompt_stats_df  : 3630 rows
  splits['train']             : 1701
  splits['val']               : 364
  splits['test_indomain']     : 365
  splits['test_heldout_domains']: 1200
  llm_judge_sample_df  : 1348 rows
  llm_judge_results_df : 3237 rows  (backends: ['claude_agent_sdk'])



[apertus1p5-sft256k-4200]
  prompts_df       : 3630 rows
  generations_df   : 36300 rows  (domains: ['aime_2025', 'codeforces', 'deepmath_103k', 'if_sft_data_verified', 'llama_nemotron', 'medical_o1', 'numinamath_1_5'])
  labels_df        : 36300 rows  (domains: ['aime_2025', 'codeforces', 'deepmath_103k', 'if_sft_data_verified', 'llama_nemotron', 'medical_o1', 'numinamath_1_5'])
  prompt_stats_df  : 3630 rows
  splits['train']             : 1701
  splits['val']               : 364
  splits['test_indomain']     : 365
  splits['test_heldout_domains']: 1200
  llm_judge_sample_df  : 1193 rows
  llm_judge_results_df : 1193 rows  (backends: ['claude_agent_sdk'])



In [9]:
# Data-integrity checks (only meaningful once prompts + splits both exist).
for tag, d in ds.items():
    print(f"[{tag}]")
    if d["prompts_ok"] and d["splits_ok"]:
        splits = d["splits"]
        prompts_df = d["prompts_df"]
        split_sets = {name: set(ids) for name, ids in splits.items()}
        pairwise_ok = True
        for i, a in enumerate(SPLIT_NAMES):
            for b in SPLIT_NAMES[i + 1:]:
                overlap = split_sets[a] & split_sets[b]
                if overlap:
                    pairwise_ok = False
                print(f"  [{'PASS' if not overlap else 'FAIL'}] {a!r} vs {b!r} disjoint: overlap = {len(overlap)}")

        held_out_ids = set(prompts_df.loc[prompts_df["held_out_domain"], "prompt_id"])
        non_heldout_ids = split_sets["train"] | split_sets["val"] | split_sets["test_indomain"]
        non_held_out_ids = set(prompts_df.loc[~prompts_df["held_out_domain"], "prompt_id"])

        leaked = held_out_ids & non_heldout_ids
        missing = held_out_ids - split_sets["test_heldout_domains"]
        stray = split_sets["test_heldout_domains"] & non_held_out_ids
        held_out_ok = not leaked and not missing and not stray

        print()
        print(f"  [{'PASS' if not leaked else 'FAIL'}] no held-out-domain prompt_id leaked into train/val/test_indomain (leaked={len(leaked)})")
        print(f"  [{'PASS' if not missing else 'FAIL'}] every held-out-domain prompt_id is present in test_heldout_domains (missing={len(missing)})")
        print(f"  [{'PASS' if not stray else 'FAIL'}] test_heldout_domains contains only held-out-domain prompt_ids (stray={len(stray)})")

        assert pairwise_ok, f"[{tag}] splits are not pairwise disjoint in prompt_id!"
        assert held_out_ok, f"[{tag}] held_out_domain prompt_ids are not exactly test_heldout_domains!"
    else:
        print("  Skipped -- prompts.parquet and/or splits/*.jsonl not present yet (see Section 3).")
    print()

[apertus-8b-instruct]
  [PASS] 'train' vs 'val' disjoint: overlap = 0
  [PASS] 'train' vs 'test_indomain' disjoint: overlap = 0
  [PASS] 'train' vs 'test_heldout_domains' disjoint: overlap = 0
  [PASS] 'val' vs 'test_indomain' disjoint: overlap = 0
  [PASS] 'val' vs 'test_heldout_domains' disjoint: overlap = 0
  [PASS] 'test_indomain' vs 'test_heldout_domains' disjoint: overlap = 0

  [PASS] no held-out-domain prompt_id leaked into train/val/test_indomain (leaked=0)
  [PASS] every held-out-domain prompt_id is present in test_heldout_domains (missing=0)
  [PASS] test_heldout_domains contains only held-out-domain prompt_ids (stray=0)

[apertus1p5-capfilter-linear-it8816]
  [PASS] 'train' vs 'val' disjoint: overlap = 0
  [PASS] 'train' vs 'test_indomain' disjoint: overlap = 0
  [PASS] 'train' vs 'test_heldout_domains' disjoint: overlap = 0
  [PASS] 'val' vs 'test_indomain' disjoint: overlap = 0
  [PASS] 'val' vs 'test_heldout_domains' disjoint: overlap = 0
  [PASS] 'test_indomain' vs 'tes

## 5. Domain / split overview

In [10]:
for tag, d in ds.items():
    print(f"[{tag}]")
    prompts_df = d["prompts_df"]
    if not prompts_df.empty:
        prompt_id_to_split = {pid: name for name, ids in d["splits"].items() for pid in ids}
        prompts_with_split = prompts_df.copy()
        prompts_with_split["split"] = prompts_with_split["prompt_id"].map(prompt_id_to_split)

        domain_split_counts = (
            prompts_with_split.groupby(["domain", "split"]).size().unstack(fill_value=0)
            .reindex(columns=SPLIT_NAMES, fill_value=0)
        )
        domain_split_counts["total"] = domain_split_counts.sum(axis=1)
        domain_split_counts = domain_split_counts.reindex(d["DOMAINS"])
        domain_split_counts.loc["all"] = domain_split_counts.sum()
        display(domain_split_counts)
    else:
        print("  prompts.parquet not available yet.")

[apertus-8b-instruct]


split,train,val,test_indomain,test_heldout_domains,total
domain,,,,,
aime_2025,21,4,5,0,30
codeforces,0,0,0,600,600
deepmath_103k,420,90,90,0,600
if_sft_data_verified,420,90,90,0,600
llama_nemotron,420,90,90,0,600
medical_o1,0,0,0,600,600
numinamath_1_5,420,90,90,0,600
all,1701,364,365,1200,3630


[apertus1p5-capfilter-linear-it8816]


split,train,val,test_indomain,test_heldout_domains,total
domain,,,,,
aime_2025,21,4,5,0,30
codeforces,0,0,0,600,600
deepmath_103k,420,90,90,0,600
if_sft_data_verified,420,90,90,0,600
llama_nemotron,420,90,90,0,600
medical_o1,0,0,0,600,600
numinamath_1_5,420,90,90,0,600
all,1701,364,365,1200,3630


[apertus1p5-sft256k-4200]


split,train,val,test_indomain,test_heldout_domains,total
domain,,,,,
aime_2025,21,4,5,0,30
codeforces,0,0,0,600,600
deepmath_103k,420,90,90,0,600
if_sft_data_verified,420,90,90,0,600
llama_nemotron,420,90,90,0,600
medical_o1,0,0,0,600,600
numinamath_1_5,420,90,90,0,600
all,1701,364,365,1200,3630


## 6. Degeneration-rate & signal distributions

`label.py` computes two kinds of signal for every rollout:

- **Per-token scores**, aligned to `generated_token_ids`:
  - **`repetition_score`** = 1 minus the *type-token ratio* (TTR) of bigrams
    in a 256-token sliding window. TTR is a standard text-diversity measure:
    "types" are the *distinct* bigrams present in the window, "tokens" are
    the *total* number of bigram positions in the window (`window_size - 1`
    of them, since each bigram spans 2 tokens). A window with no repetition
    at all has one distinct bigram per position, so types == tokens, TTR = 1,
    and `repetition_score` = 0; a window that's one bigram repeated over and
    over has only 1 distinct bigram out of many positions, so TTR is close
    to 0 and `repetition_score` is close to 1. We use **0.8** as the
    degeneration threshold: a rollout counts as degenerate when its *max*
    `repetition_score` over the sequence exceeds 0.8.
    `prompt_stats.parquet`'s `degeneration_rate` = fraction of a prompt's
    rollouts whose max `repetition_score` exceeds that threshold.
  - **`entropy`** = per-token Shannon entropy of the raw next-token
    distribution, in **nats** (natural-log units -- `-sum(p * ln(p))`,
    not `log2`; multiply by `1/ln(2)` (~1.44) to convert to bits),
    computed *before* temperature/top-p filtering (so it reflects the
    model's actual uncertainty, not the sampler's).
- **One whole-rollout LRS (Longest Repeated Substring) match**, not a
  per-token score. The intuition: scan the rollout for the longest stretch
  of tokens that occurs twice without the two copies overlapping. If the
  model is looping, that longest repeated stretch tends to cover a large
  fraction of the rollout; if it isn't, the longest thing that happens to
  repeat twice is usually short and incidental (a common phrase, a
  boilerplate line). Mechanically, `find_longest_repeated_substring`
  binary-searches over the candidate repeat length `L`: "does some
  non-overlapping pair of positions with matching `L`-token spans exist?"
  gets easier to satisfy as `L` shrinks, so a handful of probes (each an
  O(n) hash-and-bucket pass) pin down the true longest `L`. Once it has that
  one matching pair, it decomposes the matched span into its *true repeating
  unit* -- see the worked example below for why that second step matters.
  Key fields (see `degeneration_probe/dataset_gen/label.py`'s docstring for
  the full list):
  - **`lrs_length`**: tokens in the longest exact repeat found; **`lrs_score`**
    -- that length normalized by rollout length.
  - **`lrs_first_start`** / **`lrs_second_start`** / **`lrs_gap`**: where the
    two occurrences the core algorithm found start, and the distance between
    them.
  - **`lrs_period`**: the matched chunk's own smallest internal repeating
    unit, in tokens -- this is what actually distinguishes *how* a rollout
    is looping, and it can differ wildly between two rollouts whose
    `lrs_length` looks identical. Worked example, both filling a 2000-token
    rollout end to end:
    - *a single word spammed 2000 times*: the pair the binary search finds
      is `tokens[0:1000]` vs. `tokens[1000:2000]` (the longest non-overlapping
      repeat two 1000-token halves can form), so `lrs_length` = 1000 -- but
      every token inside that span is identical, so its own smallest
      internal period is 1.
    - *a 1000-token paragraph repeated exactly twice*: the matched pair is
      the same two halves, so `lrs_length` is *also* 1000 -- but this time
      the 1000-token span has no internal repetition at all, so its
      smallest period is the full 1000.

    Both hit the same `lrs_length` (1000) because that number is capped by
    how much of the rollout the repeat covers (at most `n / 2`, from
    splitting the sequence into two matching halves), not by how varied the
    repeated content is -- so a repeat spanning the whole rollout saturates
    `lrs_length` at ~`n / 2` regardless of what's inside it. `lrs_period` is
    what actually tells the two cases apart (1 vs. 1000), which is why it's
    the more useful signal for characterizing *what kind* of loop occurred.
  - **`lrs_region_starts`** / **`lrs_region_ends`** / **`lrs_period_repeat_count`**:
    once `lrs_period` is known, these describe *every* place that
    period-length unit repeats, not just the two occurrences the initial
    search happened to land on. `lrs_region_starts[i]:lrs_region_ends[i]` is
    one contiguous span of back-to-back repetition, and
    `lrs_period_repeat_count` is the unit's total repeat count summed across
    all such spans (`sum((end - start) // lrs_period for start, end in
    spans)`). There's more than one span when the unit recurs but *not*
    back-to-back -- e.g. a short catchphrase the model returns to every so
    often, with unrelated content in between each occurrence -- in which
    case each occurrence gets its own span instead of one merged range.

`rollout_signal_df` (built below) reduces each rollout's per-token arrays to
summary scalars and pulls the LRS fields through directly (already
per-rollout, no reduction needed), joined with its `num_tokens` / `stop_reason`
from `generations_df`; it's reused by Section 8's interactive explorer too.

### Locating exactly where degeneration begins (onset)

For rollouts that are degenerating, it's often useful to know not just *that*
they're degenerating but *where the repeating content first starts* -- e.g.
to train a probe on "how close is this token to the start of a repeat."
`label.py` locates this by finding the first occurrence of the LRS match
described above: `lrs_first_start` marks where the repeated span begins the
first time it appears.

That works well when the two occurrences are token-for-token identical, but
a lot of real degenerate text repeats a *template* rather than exact text --
the surrounding wording stays fixed while some number embedded in it keeps
changing: an incrementing list index (`"Case 1:"`, `"Case 2:"`, ...), a
numeral that grows by a digit each cycle (`"6"` -> `"60"` -> `"600"`, e.g. a
runaway multiplication that never resolves), or a sub-expression repeated a
growing number of times each cycle (e.g. an extra `* (1/4)` tacked on every
iteration). In all of these, an exact token-for-token match between two
occurrences either doesn't exist, or only exists for a short, incidental
stretch that isn't representative of the actual repeating pattern.

To handle this, `label.py` also computes a second version of the LRS match
where every run of consecutive digit tokens (`'0'`-`'9'`) is collapsed to a
single placeholder token before searching for repeats
(`digit_run_collapsed`), so two occurrences of the same template line up
regardless of what number appears in them or how many digits it has. This
produces a parallel set of fields, suffixed `_normalized_growing`
(`lrs_length_normalized_growing`, `lrs_first_start_normalized_growing`,
`lrs_period_normalized_growing`, `lrs_region_starts_normalized_growing` /
`lrs_region_ends_normalized_growing`, etc. -- same meanings as their
unsuffixed counterparts above, just computed on the digit-collapsed
sequence). Reported positions/content are always translated back to real
token indices and real generated text, never the placeholder values.
`lrs_first_start_normalized_growing` is the onset-position signal used
elsewhere in this project.

This isn't necessarily the final word on "where onset is" -- Section 7's
LLM judge is being extended with its own onset marker (a verbatim quote of
where it independently judges the repeat to begin) to eventually validate
and, where available, refine this signal further.


In [11]:
def _safe_int(x):
    """None/NaN-safe int() -- parquet round-trips missing ints (no-LRS-match rows) as float NaN."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    return int(x)


def build_rollout_signal_table(labels_df, generations_df):
    columns = [
        "domain", "prompt_id", "rollout_idx", "num_tokens", "stop_reason",
        "max_repetition_score", "mean_repetition_score",
        "has_lrs_match", "lrs_length", "lrs_score", "lrs_period", "lrs_period_repeat_count",
        # Digit-run-normalized onset signal -- see Section 6's "Locating exactly where
        # degeneration begins" writeup. `onset_position` is the field to use when a
        # single "where does this rollout start degenerating" number is needed.
        "has_onset_match", "onset_position", "onset_length", "onset_period", "onset_period_repeat_count",
        "mean_entropy",
    ]
    if labels_df.empty:
        return pd.DataFrame(columns=columns)

    gen_lookup = generations_df.set_index(["prompt_id", "rollout_idx"])[["num_tokens", "stop_reason"]]

    records = []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)  # all-NaN slice
        for row in labels_df.itertuples(index=False):
            repetition = np.asarray(row.repetition_score, dtype=float)
            entropy = np.asarray(row.entropy, dtype=float)
            gen_info = gen_lookup.loc[(row.prompt_id, row.rollout_idx)]
            records.append({
                "domain": row.domain,
                "prompt_id": row.prompt_id,
                "rollout_idx": int(row.rollout_idx),
                "num_tokens": int(gen_info["num_tokens"]),
                "stop_reason": gen_info["stop_reason"],
                "max_repetition_score": float(np.nanmax(repetition)) if np.any(~np.isnan(repetition)) else float("nan"),
                "mean_repetition_score": float(np.nanmean(repetition)) if np.any(~np.isnan(repetition)) else float("nan"),
                # LRS is already one match per rollout (not a per-token array) -- pulled straight
                # through, no windowed reduction needed.
                "has_lrs_match": bool(row.lrs_length > 0),
                "lrs_length": int(row.lrs_length),
                "lrs_score": float(row.lrs_score),
                "lrs_period": _safe_int(row.lrs_period),
                "lrs_period_repeat_count": _safe_int(row.lrs_period_repeat_count),
                "has_onset_match": bool(row.lrs_length_normalized_growing > 0),
                "onset_position": _safe_int(row.lrs_first_start_normalized_growing),
                "onset_length": _safe_int(row.lrs_length_normalized_growing),
                "onset_period": _safe_int(row.lrs_period_normalized_growing),
                "onset_period_repeat_count": _safe_int(row.lrs_period_repeat_count_normalized_growing),
                "mean_entropy": float(np.mean(entropy)),
            })
    return pd.DataFrame.from_records(records, columns=columns)


for tag, d in ds.items():
    rollout_signal_df = build_rollout_signal_table(d["labels_df"], d["generations_df"])
    d["rollout_signal_df"] = rollout_signal_df
    print(f"[{tag}] rollout_signal_df: {len(rollout_signal_df)} rows")

ds[DATASET_TAGS[0]]["rollout_signal_df"].head()

[apertus-8b-instruct] rollout_signal_df: 36300 rows


[apertus1p5-capfilter-linear-it8816] rollout_signal_df: 36300 rows


[apertus1p5-sft256k-4200] rollout_signal_df: 36300 rows


,domain,prompt_id,rollout_idx,num_tokens,stop_reason,max_repetition_score,mean_repetition_score,has_lrs_match,lrs_length,lrs_score,lrs_period,lrs_period_repeat_count,has_onset_match,onset_position,onset_length,onset_period,onset_period_repeat_count,mean_entropy
0,aime_2025,aime_2025_00000,0,662,eos,0.678431,0.522889,True,46,0.069486,46.0,3.0,True,318.0,51,40.0,3.0,0.222298
1,aime_2025,aime_2025_00000,1,600,eos,0.517647,0.395783,True,33,0.055000,33.0,3.0,True,57.0,33,33.0,3.0,0.348892
2,aime_2025,aime_2025_00000,2,817,eos,0.580392,0.411025,True,21,0.025704,21.0,3.0,True,539.0,21,19.0,3.0,0.421350
3,aime_2025,aime_2025_00000,3,760,eos,0.580392,0.436645,True,33,0.043421,33.0,2.0,True,56.0,33,33.0,2.0,0.239172
4,aime_2025,aime_2025_00000,4,888,eos,0.537255,0.355376,True,19,0.021396,19.0,3.0,True,562.0,26,23.0,2.0,0.421646


In [12]:
# --- repetition: prompt-level degeneration_rate (thresholded, from prompt_stats.parquet) ---
for tag, d in ds.items():
    print(f"[{tag}]")
    prompt_stats_df = d["prompt_stats_df"]
    if not prompt_stats_df.empty:
        degeneration_summary = (
            prompt_stats_df.groupby("domain")["degeneration_rate"]
            .agg(mean_degeneration_rate="mean", max_degeneration_rate="max", n_prompts="count")
        )
        display(degeneration_summary)
    else:
        print("  prompt_stats.parquet not available yet.")

[apertus-8b-instruct]


,mean_degeneration_rate,max_degeneration_rate,n_prompts
domain,,,
aime_2025,0.013333,0.1,30
codeforces,0.003667,0.2,600
deepmath_103k,0.014333,0.7,600
if_sft_data_verified,0.025500,0.7,600
llama_nemotron,0.004667,0.4,600
medical_o1,0.000000,0.0,600
numinamath_1_5,0.014167,0.3,600


[apertus1p5-capfilter-linear-it8816]


,mean_degeneration_rate,max_degeneration_rate,n_prompts
domain,,,
aime_2025,0.093333,0.3,30
codeforces,0.012667,0.4,600
deepmath_103k,0.022000,0.5,600
if_sft_data_verified,0.058333,0.9,600
llama_nemotron,0.012500,0.4,600
medical_o1,0.001667,0.1,600
numinamath_1_5,0.063500,0.8,600


[apertus1p5-sft256k-4200]


,mean_degeneration_rate,max_degeneration_rate,n_prompts
domain,,,
aime_2025,0.090000,0.4,30
codeforces,0.010333,0.3,600
deepmath_103k,0.026167,0.3,600
if_sft_data_verified,0.036833,0.9,600
llama_nemotron,0.010833,0.6,600
medical_o1,0.001000,0.2,600
numinamath_1_5,0.024667,0.5,600


In [13]:
# --- LRS: whole-rollout exact-repeat match, per domain (no fixed threshold) ---
for tag, d in ds.items():
    print(f"[{tag}]")
    rollout_signal_df = d["rollout_signal_df"]
    if not rollout_signal_df.empty:
        lrs_match_summary = (
            rollout_signal_df.groupby("domain")[["has_lrs_match", "has_onset_match"]]
            .agg(lrs_match_rate=("has_lrs_match", "mean"),
                 onset_match_rate=("has_onset_match", "mean"),
                 n_rollouts=("has_lrs_match", "count"))
        )
        display(lrs_match_summary)

        matched = rollout_signal_df[rollout_signal_df["has_lrs_match"]]
        onset_matched = rollout_signal_df[rollout_signal_df["has_onset_match"]]
        print(f"  {len(matched)}/{len(rollout_signal_df)} rollouts have an LRS repeat of >= "
              f"{label_module.DEFAULT_LRS_MIN_LENGTH} tokens.")
        print(f"  {len(onset_matched)}/{len(rollout_signal_df)} rollouts have a defined onset "
              f"position (digit-run-normalized match).")
    else:
        print("  No labeled domains available yet.")

[apertus-8b-instruct]


,lrs_match_rate,onset_match_rate,n_rollouts
domain,,,
aime_2025,0.960000,0.963333,300
codeforces,0.822000,0.838833,6000
deepmath_103k,0.902333,0.908667,6000
if_sft_data_verified,0.540000,0.584167,6000
llama_nemotron,0.825000,0.835000,6000
medical_o1,0.202333,0.211667,6000
numinamath_1_5,0.837833,0.858667,6000


  25065/36300 rollouts have an LRS repeat of >= 10 tokens.
  25711/36300 rollouts have a defined onset position (digit-run-normalized match).
[apertus1p5-capfilter-linear-it8816]


,lrs_match_rate,onset_match_rate,n_rollouts
domain,,,
aime_2025,0.983333,0.993333,300
codeforces,0.814167,0.843333,6000
deepmath_103k,0.963833,0.972167,6000
if_sft_data_verified,0.433500,0.491000,6000
llama_nemotron,0.768667,0.793500,6000
medical_o1,0.762667,0.765833,6000
numinamath_1_5,0.734167,0.761167,6000


  27157/36300 rollouts have an LRS repeat of >= 10 tokens.
  28060/36300 rollouts have a defined onset position (digit-run-normalized match).
[apertus1p5-sft256k-4200]


,lrs_match_rate,onset_match_rate,n_rollouts
domain,,,
aime_2025,0.963333,0.976667,300
codeforces,0.835333,0.860500,6000
deepmath_103k,0.912167,0.919167,6000
if_sft_data_verified,0.452333,0.506500,6000
llama_nemotron,0.785833,0.811000,6000
medical_o1,0.149500,0.152000,6000
numinamath_1_5,0.845500,0.887667,6000


  24173/36300 rollouts have an LRS repeat of >= 10 tokens.
  25114/36300 rollouts have a defined onset position (digit-run-normalized match).


In [14]:
# --- Entropy: per-rollout mean entropy distribution ---
for tag, d in ds.items():
    print(f"[{tag}]")
    rollout_signal_df = d["rollout_signal_df"]
    if not rollout_signal_df.empty:
        entropy_summary = (
            rollout_signal_df.groupby("domain")["mean_entropy"]
            .agg(mean="mean", max="max", min="min")
        )
        display(entropy_summary)
    else:
        print("  No labeled domains available yet.")

[apertus-8b-instruct]


,mean,max,min
domain,,,
aime_2025,0.427700,0.966822,0.016300
codeforces,0.358503,1.004798,0.009439
deepmath_103k,0.414214,1.317496,0.012662
if_sft_data_verified,0.617717,4.593693,0.004877
llama_nemotron,0.435042,1.097740,0.006070
medical_o1,0.818040,1.641292,0.073178
numinamath_1_5,0.373331,1.347791,0.006872


[apertus1p5-capfilter-linear-it8816]


,mean,max,min
domain,,,
aime_2025,0.174721,0.491418,0.031722
codeforces,0.251631,0.690874,0.009281
deepmath_103k,0.270388,1.045154,0.030188
if_sft_data_verified,0.621972,4.356446,0.002772
llama_nemotron,0.256914,1.077006,0.010899
medical_o1,0.564922,3.253996,0.018185
numinamath_1_5,0.330944,1.866546,0.007937


[apertus1p5-sft256k-4200]


,mean,max,min
domain,,,
aime_2025,0.260344,1.007261,0.032535
codeforces,0.256781,0.752959,0.009742
deepmath_103k,0.350859,1.836448,0.019784
if_sft_data_verified,0.643011,4.966056,0.001506
llama_nemotron,0.248608,0.799281,0.016623
medical_o1,0.924167,3.539158,0.017115
numinamath_1_5,0.316581,1.970370,0.013782


In [15]:
# --- length-cap rate: fraction of rollouts that hit max_new_tokens without EOS ---
for tag, d in ds.items():
    print(f"[{tag}]")
    generations_df = d["generations_df"]
    if not generations_df.empty:
        length_cap_summary = (
            generations_df.assign(hit_cap=generations_df["stop_reason"] == "length")
            .groupby("domain")["hit_cap"]
            .mean()
            .rename("length_cap_rate")
        )
        display(length_cap_summary)
    else:
        print("  No generated domains available yet.")

[apertus-8b-instruct]


domain
aime_2025               0.033333
codeforces              0.004167
deepmath_103k           0.026833
if_sft_data_verified    0.060667
llama_nemotron          0.020500
medical_o1              0.000167
numinamath_1_5          0.034333
Name: length_cap_rate, dtype: float64

[apertus1p5-capfilter-linear-it8816]


domain
aime_2025               0.056667
codeforces              0.016667
deepmath_103k           0.014833
if_sft_data_verified    0.092667
llama_nemotron          0.021333
medical_o1              0.002500
numinamath_1_5          0.073833
Name: length_cap_rate, dtype: float64

[apertus1p5-sft256k-4200]


domain
aime_2025               0.133333
codeforces              0.018667
deepmath_103k           0.024000
if_sft_data_verified    0.066333
llama_nemotron          0.021333
medical_o1              0.012667
numinamath_1_5          0.049167
Name: length_cap_rate, dtype: float64

In [16]:
# --- Does hitting the token cap correlate with degeneration? ---
# Validated finding: hitting the 4096-token generation cap is almost always
# degeneration, not just a long-but-healthy response -- nearly every capped
# rollout has a massive LRS repeat (often a third or more of the whole
# response), vs. a small, likely-benign incidental repeat for normally
# (eos-)terminated rollouts.
for tag, d in ds.items():
    print(f"[{tag}]")
    rollout_signal_df = d["rollout_signal_df"]
    if not rollout_signal_df.empty:
        hit_cap = rollout_signal_df["stop_reason"] == "length"
        cap_label = hit_cap.map({False: "eos", True: "hit_cap (length)"})

        crosstab = pd.DataFrame({
            "n_rollouts": rollout_signal_df.groupby(cap_label).size(),
            "lrs_match_rate": rollout_signal_df.groupby(cap_label)["has_lrs_match"].mean(),
            "mean_max_repetition_score": rollout_signal_df.groupby(cap_label)["max_repetition_score"].mean(),
        })
        display(crosstab)

        matched = rollout_signal_df[rollout_signal_df["has_lrs_match"]]
        if not matched.empty:
            matched_cap_label = cap_label.loc[matched.index]
            print("  Among matched rollouts, median lrs_length / lrs_period by stop_reason:")
            display(
                matched.groupby(matched_cap_label)[["lrs_length", "lrs_period"]].median()
            )
    else:
        print("  No labeled domains available yet.")

[apertus-8b-instruct]


,n_rollouts,lrs_match_rate,mean_max_repetition_score
stop_reason,,,
eos,35410,0.682773,0.419100
hit_cap (length),890,0.997753,0.693558


  Among matched rollouts, median lrs_length / lrs_period by stop_reason:


,lrs_length,lrs_period
stop_reason,,
eos,19.0,19.0
hit_cap (length),1528.5,177.0


[apertus1p5-capfilter-linear-it8816]


...

,lrs_length,lrs_period
stop_reason,,
eos,21.0,21.0
hit_cap (length),1672.0,53.0


[apertus1p5-sft256k-4200]


,n_rollouts,lrs_match_rate,mean_max_repetition_score
stop_reason,,,
eos,35107,0.654599,0.476490
hit_cap (length),1193,0.999162,0.749611


  Among matched rollouts, median lrs_length / lrs_period by stop_reason:


,lrs_length,lrs_period
stop_reason,,
eos,19.0,19.0
hit_cap (length),1420.0,61.0


### 6.1 Cross-dataset comparison

The three builds (Section 1) are otherwise directly comparable (same domains,
same prompts, same `max_new_tokens=4096` generation budget) -- this puts them
side by side on the same heuristic signals as above, dataset as columns
instead of dataset as a separate loop iteration per cell.

**The caveat matters more than the numbers below.** Only `apertus-8b-instruct`
has any LLM-judge calibration data (Section 7) yet, and even that is partial
(956/1219 judged so far). Everything in this subsection is heuristic-only
(`repetition_score`, LRS, length-cap) -- exactly the signals Section 7 shows
we can't fully trust yet. A dataset looking "more degenerate" here could mean
it genuinely loops more often, or could mean it does more of the kind of
legitimate long-form enumeration (step-by-step math derivations, case-by-case
reasoning) that inflates these heuristics without being real degeneration --
the per-domain breakdown below is precisely where that ambiguity shows up
(the gap is concentrated in the math-heavy domains), and we can't resolve it
one way or the other until the LLM judge has run on these datasets too.

In [17]:
# --- cross-dataset aggregate summary (heuristic-only -- see 6.1's caveat above) ---
agg_rows = {}
for tag, d in ds.items():
    rollout_signal_df = d["rollout_signal_df"]
    if rollout_signal_df.empty:
        continue
    agg_rows[tag] = {
        "n_rollouts": len(rollout_signal_df),
        "repetition_degenerate_rate": (
            rollout_signal_df["max_repetition_score"] > label_module.DEFAULT_DEGENERATION_THRESHOLD
        ).mean(),
        "lrs_match_rate (plain)": rollout_signal_df["has_lrs_match"].mean(),
        "lrs_match_rate (onset, digit-norm)": rollout_signal_df["has_onset_match"].mean(),
        "length_cap_rate": (rollout_signal_df["stop_reason"] == "length").mean(),
        "mean_entropy": rollout_signal_df["mean_entropy"].mean(),
    }
cross_dataset_summary = pd.DataFrame(agg_rows).T
cross_dataset_summary.index.name = "dataset"
display(cross_dataset_summary.style.format({
    "n_rollouts": "{:,.0f}",
    "repetition_degenerate_rate": "{:.2%}",
    "lrs_match_rate (plain)": "{:.2%}",
    "lrs_match_rate (onset, digit-norm)": "{:.2%}",
    "length_cap_rate": "{:.2%}",
    "mean_entropy": "{:.3f}",
}))

,n_rollouts,repetition_degenerate_rate,lrs_match_rate (plain),"lrs_match_rate (onset, digit-norm)",length_cap_rate,mean_entropy
dataset,,,,,,
apertus-8b-instruct,"36,300",1.04%,69.05%,70.83%,2.45%,0.502
apertus1p5-capfilter-linear-it8816,"36,300",2.90%,74.81%,77.30%,3.71%,0.381
apertus1p5-sft256k-4200,"36,300",1.89%,66.59%,69.18%,3.29%,0.455


In [18]:
# --- cross-dataset per-domain breakdown: repetition_score-based degenerate rate ---
domain_rep_rows = []
for tag, d in ds.items():
    rollout_signal_df = d["rollout_signal_df"]
    if rollout_signal_df.empty:
        continue
    s = (
        rollout_signal_df
        .assign(is_rep_degen=rollout_signal_df["max_repetition_score"] > label_module.DEFAULT_DEGENERATION_THRESHOLD)
        .groupby("domain")["is_rep_degen"].mean()
    )
    domain_rep_rows.append(s.rename(tag))
domain_rep_pivot = pd.concat(domain_rep_rows, axis=1)
print(f"repetition-degenerate rate (max_repetition_score > {label_module.DEFAULT_DEGENERATION_THRESHOLD}), by domain and dataset:")
display(domain_rep_pivot.style.format("{:.2%}"))

repetition-degenerate rate (max_repetition_score > 0.8), by domain and dataset:


,apertus-8b-instruct,apertus1p5-capfilter-linear-it8816,apertus1p5-sft256k-4200
domain,,,
aime_2025,1.33%,9.33%,9.00%
codeforces,0.37%,1.27%,1.03%
deepmath_103k,1.43%,2.20%,2.62%
if_sft_data_verified,2.55%,5.83%,3.68%
llama_nemotron,0.47%,1.25%,1.08%
medical_o1,0.00%,0.17%,0.10%
numinamath_1_5,1.42%,6.35%,2.47%


In [19]:
# --- cross-dataset per-domain breakdown: length-cap (truncation) rate ---
domain_cap_rows = []
for tag, d in ds.items():
    rollout_signal_df = d["rollout_signal_df"]
    if rollout_signal_df.empty:
        continue
    s = (
        rollout_signal_df
        .assign(hit_cap=rollout_signal_df["stop_reason"] == "length")
        .groupby("domain")["hit_cap"].mean()
    )
    domain_cap_rows.append(s.rename(tag))
domain_cap_pivot = pd.concat(domain_cap_rows, axis=1)
print("length-cap (truncation) rate, by domain and dataset:")
display(domain_cap_pivot.style.format("{:.2%}"))

length-cap (truncation) rate, by domain and dataset:


,apertus-8b-instruct,apertus1p5-capfilter-linear-it8816,apertus1p5-sft256k-4200
domain,,,
aime_2025,3.33%,5.67%,13.33%
codeforces,0.42%,1.67%,1.87%
deepmath_103k,2.68%,1.48%,2.40%
if_sft_data_verified,6.07%,9.27%,6.63%
llama_nemotron,2.05%,2.13%,2.13%
medical_o1,0.02%,0.25%,1.27%
numinamath_1_5,3.43%,7.38%,4.92%


## 7. LLM-judge calibration

The heuristics above (`repetition_score`'s 0.8 threshold, `lrs_period_repeat_count`)
are cheap proxies for "this rollout is degenerating." `llm_judge.py` asks an LLM to
independently judge a targeted sample of rollouts, blind to the heuristic scores,
seeing only the prompt + completion text, so those thresholds can be checked
against something closer to ground truth.

### Which rollouts get judged

Every rollout that hit `max_new_tokens` (4096) without reaching EOS
(`stratum == "truncated"`), computed once by `llm_judge.select_calibration_sample`
and written to `llm_judge/calibration_sample.parquet` (loaded in Section 4 as
`llm_judge_sample_df`) so the sample is fixed and reproducible across
backends/reruns. This is the population most likely to contain real degeneration
loops -- `stop_reason == "length"` alone was already validated at 99.6% precision
against LLM-judge ground truth (see `onset_labels.py`'s module docstring), so
rollouts that reached EOS on their own are never judged, regardless of heuristic
score.

(Earlier versions of this pipeline also judged a `flagged_natural_eos` stratum --
EOS-terminated rollouts the heuristics flagged anyway -- to check for false
positives. That stratum was dropped: not worth the judge-call budget once
`stop_reason == "length"` precision was established directly.)

### Backends

Three interchangeable `JudgeBackend` implementations, swap with `--backend`,
no code change (see `degeneration_probe/dataset_gen/llm_judge.py`):

- **`anthropic`**: direct Messages API, pay-per-token, native structured
  outputs (`output_config.format`), no parse-retry loop needed.
- **`claude_agent_sdk`**: routes through the real `claude` CLI subprocess
  (`claude_agent_sdk.query`) using a `claude setup-token` OAuth token, so
  calls draw on Pro/Max subscription usage instead of metered billing.
  `tools=[]` disables all built-in tools, this is a one-shot text
  judgment, not an agentic session. If subscription usage runs out mid-batch,
  the run stops cleanly (`UsageExhausted`) instead of burning through every
  remaining row as an instant failure, rerunning the same command later
  resumes once usage resets.
- **`openrouter`**: OpenAI-compatible endpoint fronting open models; not
  every model reliably honors a strict JSON-schema `response_format`, so it
  falls back to prompting for JSON and retrying on parse failure. Rate
  limits (common on free-tier models) are retried with exponential backoff.

#### Setting up `claude_agent_sdk` credentials

`ClaudeAgentSDKBackend` reads its OAuth token from
`~/keys/.anthropic_oauth_token` (a plain text file containing just the
token). To generate one:

```
claude setup-token
```

This walks through a browser login against your Pro/Max subscription and
prints the resulting token to the terminal. Save it where the backend
expects it, matching this repo's existing convention of keeping credentials
under `~/keys/` (alongside `~/keys/.hf_token`, `~/keys/.openrouter_key`):

```
mkdir -p ~/keys
echo "<paste the token here>" > ~/keys/.anthropic_oauth_token
chmod 600 ~/keys/.anthropic_oauth_token
```

No other configuration is needed -- `--backend claude_agent_sdk` picks this
file up automatically (`Path.home() / "keys" / ".anthropic_oauth_token"`).

Results are a resumable, per-backend manifest (`llm_judge/results_<backend>.parquet`,
loaded in Section 4 as `llm_judge_results_df`), rerunning the same command
skips every rollout already recorded `status="ok"`, so an interrupted run
just needs to be rerun later to resume.

To run/resume judging with a single backend/account directly:

```
.venv/bin/python -m degeneration_probe.dataset_gen.llm_judge --config <cfg> --backend claude_agent_sdk
```

#### Running the full-scale judge job: `judge.sbatch` and multi-account rotation

The full calibration sample (~1200 rollouts) is more `claude_agent_sdk` usage than
a single Pro/Max subscription's usage window comfortably covers in one run, so
the full-scale build doesn't call `llm_judge.py` directly -- it submits
`cluster/utils/dataset/judge.sbatch`, which rotates across *several* accounts'
OAuth tokens in sequence instead of just one:

```
sbatch cluster/utils/dataset/judge.sbatch
```

Mechanically: the script hardcodes a `TOKEN_NAMES` list (currently
`marcodenegri`, `lucasartori`, `lorenzobaggi`) and, for each name in order,
looks for a key file at `~/keys/.anthropic_oauth_token_<name>` -- note the
`_<name>` suffix, *different* from the single default file
`~/keys/.anthropic_oauth_token` described above, since here every account
needs its own file to rotate between them. For each account with a key file
present, it invokes `llm_judge.py --oauth-key-file ~/keys/.anthropic_oauth_token_<name>`
(the flag that lets a caller point at a specific account's token instead of
the default file). When that account's subscription usage runs out,
`UsageExhausted` makes the call exit cleanly rather than fail, and the loop
moves on to the next account's key file; after each account it also checks
how many calibration rows are still not `status="ok"` and stops the whole
job early once none remain, instead of pointlessly starting the next
account. Since everything resumes into the same
`llm_judge/results_claude_agent_sdk.parquet` (skipping rows already `"ok"`,
same as running `llm_judge.py` directly), the job is safe to resubmit as-is
whenever more accounts' usage windows have reset.

In [20]:
print("System prompt sent to the judge (blind -- prompt + completion text only, no heuristic scores):")
print(textwrap.indent(llm_judge_module.JUDGE_SYSTEM_PROMPT, "  "))
print()
print("Requested output schema (JudgeVerdict -- returned as structured JSON, not free text):")
for field_name, field_info in llm_judge_module.JudgeVerdict.model_fields.items():
    print(f"  {field_name}: {field_info.description}")

System prompt sent to the judge (blind -- prompt + completion text only, no heuristic scores):
  You are judging whether a language model's completion degenerated into repetitive, low-information text (looping on a phrase, sentence, or pattern without making progress), as opposed to repetition that is a normal, expected part of the task. You will see both the prompt and the completion -- read the prompt first, since it changes what counts as degeneration.

  Two things that look repetitive but are NOT degeneration:

  1. Genuine incremental work. Brute-force search, case enumeration, or a step-by-step derivation often produces a fixed template with a changing value each time (e.g. "N = 41: ..." then "N = 42: ..." then "N = 43: ..."). This is fine as long as each occurrence is doing new work -- checking a new candidate, advancing to the next step, covering a new case. A useful test: if you deleted every occurrence but the first, would the completion lose information needed to solve the 

In [21]:
# --- Judge vs. heuristic agreement (only meaningful for status="ok" rows) ---
for tag, d in ds.items():
    print(f"[{tag}]")
    llm_judge_results_df = d["llm_judge_results_df"]
    llm_judge_sample_df = d["llm_judge_sample_df"]
    rollout_signal_df = d["rollout_signal_df"]
    if not llm_judge_results_df.empty:
        ok = llm_judge_results_df[llm_judge_results_df["status"] == "ok"].merge(
            llm_judge_sample_df[["prompt_id", "rollout_idx", "stratum"]],
            on=["prompt_id", "rollout_idx"], how="left",
        ).merge(
            rollout_signal_df[["prompt_id", "rollout_idx", "max_repetition_score", "lrs_period_repeat_count"]],
            on=["prompt_id", "rollout_idx"], how="left",
        )
        ok["heuristic_flag"] = (
            (ok["max_repetition_score"] > llm_judge_module.DEGENERATION_THRESHOLD)
            | (ok["lrs_period_repeat_count"] >= llm_judge_module.MIN_PERIOD_REPEAT_COUNT)
        )
        ok["agree"] = ok["is_degenerating"] == ok["heuristic_flag"]

        agreement = ok.groupby(["backend", "stratum"]).agg(
            n=("is_degenerating", "size"),
            judge_flagged_rate=("is_degenerating", "mean"),
            heuristic_flagged_rate=("heuristic_flag", "mean"),
            agreement_rate=("agree", "mean"),
        )
        display(agreement)

        failed = llm_judge_results_df[llm_judge_results_df["status"] == "failed"]
        if not failed.empty:
            print(f"  {len(failed)} judge call(s) failed (see 'error' column), e.g.:")
            display(failed[["prompt_id", "rollout_idx", "backend", "error"]].head(5))
    else:
        print("  No judge results yet -- run llm_judge.py with a backend first (see command above).")

[apertus-8b-instruct]


,,n,judge_flagged_rate,heuristic_flagged_rate,agreement_rate
backend,stratum,,,,
claude_agent_sdk,truncated,827,0.990326,0.939541,0.932285


  964 judge call(s) failed (see 'error' column), e.g.:


,prompt_id,rollout_idx,backend,error
0,aime_2025_00004,1,anthropic,"Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is to..."
1,aime_2025_00009,5,anthropic,"Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is to..."
2,aime_2025_00010,4,anthropic,"Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is to..."
3,aime_2025_00015,9,anthropic,"Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is to..."
4,aime_2025_00016,5,anthropic,"Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is to..."


[apertus1p5-capfilter-linear-it8816]


...

,prompt_id,rollout_idx,backend,error
122,aime_2025_00016,2,claude_agent_sdk,"claude_agent_sdk judge failed (subtype=error_max_structured_output_retries, stop_reason=tool_use, errors=['Failed to..."
125,aime_2025_00022,0,claude_agent_sdk,"claude_agent_sdk judge failed (subtype=error_max_structured_output_retries, stop_reason=tool_use, errors=['Failed to..."
133,deepmath_103k_00001,9,claude_agent_sdk,"claude_agent_sdk judge failed (subtype=error_max_structured_output_retries, stop_reason=tool_use, errors=['Failed to..."
134,deepmath_103k_00004,0,claude_agent_sdk,"claude_agent_sdk judge failed (subtype=error_max_structured_output_retries, stop_reason=tool_use, errors=['Failed to..."
136,deepmath_103k_00004,7,claude_agent_sdk,"claude_agent_sdk judge failed (subtype=error_max_structured_output_retries, stop_reason=tool_use, errors=['Failed to..."


[apertus1p5-sft256k-4200]


,,n,judge_flagged_rate,heuristic_flagged_rate,agreement_rate
backend,stratum,,,,
claude_agent_sdk,truncated,1111,0.929793,0.816382,0.836184


  82 judge call(s) failed (see 'error' column), e.g.:


,prompt_id,rollout_idx,backend,error
1095,if_sft_data_verified_00003,2,claude_agent_sdk,"claude_agent_sdk judge failed (subtype=success, stop_reason=refusal, errors=None): result='API Error: Claude Code is..."
1096,if_sft_data_verified_00003,3,claude_agent_sdk,"claude_agent_sdk judge failed (subtype=success, stop_reason=refusal, errors=None): result='API Error: Claude Code is..."
1097,if_sft_data_verified_00025,5,claude_agent_sdk,"claude_agent_sdk judge failed (subtype=success, stop_reason=refusal, errors=None): result='API Error: Claude Code is..."
1098,if_sft_data_verified_00029,0,claude_agent_sdk,"claude_agent_sdk judge failed (subtype=success, stop_reason=refusal, errors=None): result='API Error: Claude Code is..."
1099,if_sft_data_verified_00029,3,claude_agent_sdk,"claude_agent_sdk judge failed (subtype=success, stop_reason=refusal, errors=None): result='API Error: Claude Code is..."


In [22]:
# --- Confusion matrix: does LRS alone (lrs_period_repeat_count >= min, i.e. the
# repeating unit recurs at least this many times) correctly flag what the judge calls
# degenerating, at completion level? Deliberately LRS-only (no repetition_score/TTR
# mixed in), to keep this an evaluation of LRS specifically -- see the match-rate
# discussion below for LRS's weaker, "found any repeat at all" signal by contrast.
# Restricted to stratum="truncated" -- the only stratum still actively judged (see
# Section 7 markdown; "flagged_natural_eos" is a legacy stratum from an earlier
# pipeline version, not representative of current judging).
for tag, d in ds.items():
    print(f"[{tag}]")
    llm_judge_results_df = d["llm_judge_results_df"]
    llm_judge_sample_df = d["llm_judge_sample_df"]
    rollout_signal_df = d["rollout_signal_df"]
    if llm_judge_results_df.empty:
        print("  No judge results yet.")
        continue

    ok = llm_judge_results_df[llm_judge_results_df["status"] == "ok"].merge(
        llm_judge_sample_df[["prompt_id", "rollout_idx", "domain", "stratum"]],
        on=["prompt_id", "rollout_idx"], how="left",
    ).merge(
        rollout_signal_df[["prompt_id", "rollout_idx", "lrs_period_repeat_count"]],
        on=["prompt_id", "rollout_idx"], how="left",
    )
    ok = ok[ok["stratum"] == "truncated"]
    if ok.empty:
        print("  No 'truncated'-stratum judge results yet.")
        continue

    ok["heuristic_flag"] = ok["lrs_period_repeat_count"] >= llm_judge_module.MIN_PERIOD_REPEAT_COUNT

    def _confusion_row(g):
        judge, heur = g["is_degenerating"], g["heuristic_flag"]
        tp, fp = int((judge & heur).sum()), int((~judge & heur).sum())
        fn, tn = int((judge & ~heur).sum()), int((~judge & ~heur).sum())
        n = len(g)
        return pd.Series({
            "n": n, "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "precision": tp / (tp + fp) if (tp + fp) else float("nan"),
            "recall": tp / (tp + fn) if (tp + fn) else float("nan"),
            "accuracy": (tp + tn) / n if n else float("nan"),
        })

    per_domain = ok.groupby("domain", group_keys=True).apply(_confusion_row, include_groups=False)
    total = _confusion_row(ok).to_frame("ALL domains").T
    display(pd.concat([per_domain, total]))


[apertus-8b-instruct]


,n,tp,fp,fn,tn,precision,recall,accuracy
aime_2025,10.0,10.0,0.0,0.0,0.0,1.000000,1.000000,1.000000
codeforces,25.0,22.0,0.0,3.0,0.0,1.000000,0.880000,0.880000
deepmath_103k,161.0,152.0,4.0,4.0,1.0,0.974359,0.974359,0.950311
if_sft_data_verified,302.0,289.0,0.0,13.0,0.0,1.000000,0.956954,0.956954
llama_nemotron,123.0,98.0,0.0,25.0,0.0,1.000000,0.796748,0.796748
medical_o1,1.0,1.0,0.0,0.0,0.0,1.000000,1.000000,1.000000
numinamath_1_5,205.0,191.0,3.0,11.0,0.0,0.984536,0.945545,0.931707
ALL domains,827.0,763.0,7.0,56.0,1.0,0.990909,0.931624,0.923821


[apertus1p5-capfilter-linear-it8816]


,n,tp,fp,fn,tn,precision,recall,accuracy
aime_2025,17.0,16.0,0.0,1.0,0.0,1.000000,0.941176,0.941176
codeforces,100.0,93.0,0.0,7.0,0.0,1.000000,0.930000,0.930000
deepmath_103k,89.0,79.0,0.0,10.0,0.0,1.000000,0.887640,0.887640
if_sft_data_verified,489.0,456.0,1.0,30.0,2.0,0.997812,0.938272,0.936605
llama_nemotron,128.0,119.0,0.0,9.0,0.0,1.000000,0.929688,0.929688
medical_o1,15.0,14.0,0.0,1.0,0.0,1.000000,0.933333,0.933333
numinamath_1_5,440.0,422.0,2.0,15.0,1.0,0.995283,0.965675,0.961364
ALL domains,1278.0,1199.0,3.0,73.0,3.0,0.997504,0.942610,0.940532


[apertus1p5-sft256k-4200]


,n,tp,fp,fn,tn,precision,recall,accuracy
aime_2025,40.0,21.0,1.0,8.0,10.0,0.954545,0.724138,0.775000
codeforces,112.0,104.0,0.0,8.0,0.0,1.000000,0.928571,0.928571
deepmath_103k,144.0,120.0,2.0,19.0,3.0,0.983607,0.863309,0.854167
if_sft_data_verified,319.0,285.0,6.0,21.0,7.0,0.979381,0.931373,0.915361
llama_nemotron,127.0,118.0,0.0,9.0,0.0,1.000000,0.929134,0.929134
medical_o1,76.0,61.0,0.0,15.0,0.0,1.000000,0.802632,0.802632
numinamath_1_5,293.0,150.0,19.0,94.0,30.0,0.887574,0.614754,0.614334
ALL domains,1111.0,859.0,28.0,174.0,50.0,0.968433,0.831559,0.818182


In [23]:
# --- onset_quote validation: can the LLM's claimed onset_quote actually be found,
# verbatim, in the rollout it was judging? A wrong quote (paraphrased, or copied from
# the wrong part of the completion) would silently poison onset_quote as an onset-position
# signal (see onset_labels.py's resolve_onset_position), so check the raw hit rate here
# before trusting it at scale. Reuses `find_string_in_tokens` -- the same binary-search
# lookup `resolve_onset_position(metric="onset_quote")` uses in production.
from degeneration_probe.utils.tokenization import find_string_in_tokens
import torch

for tag, d in ds.items():
    print(f"[{tag}]")
    llm_judge_results_df = d["llm_judge_results_df"]
    generations_df = d["generations_df"]
    if llm_judge_results_df.empty:
        print("  No judge results yet.")
        d["onset_quote_match_df"] = pd.DataFrame()
        continue

    tokenizer = get_tokenizer(tag)
    gen_lookup = generations_df.set_index(["prompt_id", "rollout_idx"])["generated_token_ids"]

    quoted = llm_judge_results_df[
        (llm_judge_results_df["status"] == "ok")
        & llm_judge_results_df["onset_quote"].notna()
        & (llm_judge_results_df["onset_quote"] != "")
    ]

    records = []
    for row in quoted.itertuples(index=False):
        token_ids = torch.as_tensor([int(t) for t in gen_lookup.loc[(row.prompt_id, row.rollout_idx)]])
        try:
            span = find_string_in_tokens(row.onset_quote, token_ids, tokenizer)
            matched, llm_onset_position = True, int(span.start)
        except (AssertionError, ValueError):
            matched, llm_onset_position = False, None
        records.append({
            "backend": row.backend,
            "prompt_id": row.prompt_id,
            "rollout_idx": int(row.rollout_idx),
            "onset_quote": row.onset_quote,
            "matched": matched,
            "llm_onset_position": llm_onset_position,
        })
    onset_quote_match_df = pd.DataFrame.from_records(records)
    d["onset_quote_match_df"] = onset_quote_match_df

    if onset_quote_match_df.empty:
        print("  No onset_quote values to validate yet (no 'ok' judge results with a non-null onset_quote).")
        continue

    match_summary = onset_quote_match_df.groupby("backend").agg(
        n=("matched", "size"), n_matched=("matched", "sum"), match_rate=("matched", "mean"),
    )
    display(match_summary)

    mismatches = onset_quote_match_df[~onset_quote_match_df["matched"]]
    if not mismatches.empty:
        print(f"  {len(mismatches)} onset_quote(s) NOT found verbatim in their rollout's text -- a sample:")
        display(mismatches[["backend", "prompt_id", "rollout_idx", "onset_quote"]].head(10))


[apertus-8b-instruct]


,n,n_matched,match_rate
backend,,,
claude_agent_sdk,961,959,0.997919


  2 onset_quote(s) NOT found verbatim in their rollout's text -- a sample:


,backend,prompt_id,rollout_idx,onset_quote
585,claude_agent_sdk,llama_nemotron_00249,7,This should be the correct solution.\n\n\n```python\nimport sys\nfrom collections import deque
716,claude_agent_sdk,numinamath_1_5_00168,2,"\[ \boxed{0 \leq x + y \leq 4} \][{""display_answers"": {""answers"": ["


[apertus1p5-capfilter-linear-it8816]


,n,n_matched,match_rate
backend,,,
claude_agent_sdk,1475,1469,0.995932


  6 onset_quote(s) NOT found verbatim in their rollout's text -- a sample:


,backend,prompt_id,rollout_idx,onset_quote
106,claude_agent_sdk,deepmath_103k_00173,9,N/A
135,claude_agent_sdk,deepmath_103k_00240,6,Compute f(4) again
157,claude_agent_sdk,deepmath_103k_00298,8,test
166,claude_agent_sdk,deepmath_103k_00323,4,none
377,claude_agent_sdk,if_sft_data_verified_00151,9,"The problem contains an inconsistency in the initial conditions, but we have solved the equations as given.\n\n\n\nL..."
748,claude_agent_sdk,if_sft_data_verified_00392,9,"In summary, the height is O(log n) because each step reduces the problem size by a factor of 3, leading to a logarit..."


[apertus1p5-sft256k-4200]


,n,n_matched,match_rate
backend,,,
claude_agent_sdk,1033,1033,1.0


In [24]:
# --- onset-position agreement: for rollouts where onset_quote was actually found (cell
# above), how far apart (in tokens) is the LLM's onset_quote position from the
# digit-run-normalized LRS onset position (Section 6)? Only computed where BOTH signals
# are defined: a matched onset_quote AND a defined LRS onset (`has_onset_match`) --
# rollouts failing either check have no meaningful "distance" to report.
for tag, d in ds.items():
    print(f"[{tag}]")
    onset_quote_match_df = d.get("onset_quote_match_df", pd.DataFrame())
    rollout_signal_df = d["rollout_signal_df"]
    if onset_quote_match_df.empty or rollout_signal_df.empty:
        print("  Nothing to compare yet.")
        continue

    matched = onset_quote_match_df[onset_quote_match_df["matched"]]
    merged = matched.merge(
        rollout_signal_df[["prompt_id", "rollout_idx", "has_onset_match", "onset_position"]],
        on=["prompt_id", "rollout_idx"], how="left",
    )
    merged = merged[merged["has_onset_match"] == True]
    if merged.empty:
        print("  No rollouts with both a matched onset_quote and a defined LRS onset position.")
        continue

    merged["token_distance"] = merged["llm_onset_position"] - merged["onset_position"]
    merged["abs_token_distance"] = merged["token_distance"].abs()

    distance_summary = merged.groupby("backend").agg(
        n=("abs_token_distance", "size"),
        mean_abs_token_distance=("abs_token_distance", "mean"),
        median_abs_token_distance=("abs_token_distance", "median"),
        mean_signed_token_distance=("token_distance", "mean"),  # >0: LLM's onset is later than LRS's
    )
    display(distance_summary)


[apertus-8b-instruct]


,n,mean_abs_token_distance,median_abs_token_distance,mean_signed_token_distance
backend,,,,
claude_agent_sdk,956,267.639121,92.0,-93.896444


[apertus1p5-capfilter-linear-it8816]


,n,mean_abs_token_distance,median_abs_token_distance,mean_signed_token_distance
backend,,,,
claude_agent_sdk,1466,175.287176,29.0,-47.184857


[apertus1p5-sft256k-4200]


,n,mean_abs_token_distance,median_abs_token_distance,mean_signed_token_distance
backend,,,,
claude_agent_sdk,1031,379.945684,70.0,-100.320078


In [25]:
# --- onset-position agreement (same signal as the cell above), broken down by domain and
# totaled, to see whether onset agreement differs by domain rather than only by backend.
for tag, d in ds.items():
    print(f"[{tag}]")
    onset_quote_match_df = d.get("onset_quote_match_df", pd.DataFrame())
    rollout_signal_df = d["rollout_signal_df"]
    llm_judge_sample_df = d["llm_judge_sample_df"]
    if onset_quote_match_df.empty or rollout_signal_df.empty:
        print("  Nothing to compare yet.")
        continue

    matched = onset_quote_match_df[onset_quote_match_df["matched"]]
    merged = matched.merge(
        rollout_signal_df[["prompt_id", "rollout_idx", "has_onset_match", "onset_position"]],
        on=["prompt_id", "rollout_idx"], how="left",
    ).merge(
        llm_judge_sample_df[["prompt_id", "rollout_idx", "domain"]],
        on=["prompt_id", "rollout_idx"], how="left",
    )
    merged = merged[merged["has_onset_match"] == True]
    if merged.empty:
        print("  No rollouts with both a matched onset_quote and a defined LRS onset position.")
        continue

    merged["token_distance"] = merged["llm_onset_position"] - merged["onset_position"]
    merged["abs_token_distance"] = merged["token_distance"].abs()

    def _distance_row(g):
        return pd.Series({
            "n": len(g),
            "mean_abs_token_distance": g["abs_token_distance"].mean(),
            "median_abs_token_distance": g["abs_token_distance"].median(),
            "mean_signed_token_distance": g["token_distance"].mean(),
        })

    per_domain = merged.groupby("domain", group_keys=True).apply(_distance_row, include_groups=False)
    total = _distance_row(merged).to_frame("ALL domains").T
    display(pd.concat([per_domain, total]))


[apertus-8b-instruct]


,n,mean_abs_token_distance,median_abs_token_distance,mean_signed_token_distance
aime_2025,10.0,353.400000,209.0,-329.800000
codeforces,24.0,293.958333,50.5,-192.958333
deepmath_103k,155.0,207.522581,102.0,-96.800000
if_sft_data_verified,301.0,256.212625,75.0,-116.817276
llama_nemotron,123.0,583.853659,264.0,-274.910569
medical_o1,1.0,70.000000,70.0,70.000000
numinamath_1_5,201.0,228.393035,113.0,-13.815920
ALL domains,956.0,267.639121,92.0,-93.896444


[apertus1p5-capfilter-linear-it8816]


,n,mean_abs_token_distance,median_abs_token_distance,mean_signed_token_distance
aime_2025,17.0,240.058824,82.0,-27.470588
codeforces,100.0,328.580000,47.0,-235.580000
deepmath_103k,89.0,314.224719,52.0,-68.314607
if_sft_data_verified,482.0,131.769710,28.5,7.080913
llama_nemotron,128.0,288.890625,29.0,-130.937500
medical_o1,15.0,212.600000,13.0,-74.466667
numinamath_1_5,436.0,137.830275,23.0,-81.165138
ALL domains,1466.0,175.287176,29.0,-47.184857


[apertus1p5-sft256k-4200]


,n,mean_abs_token_distance,median_abs_token_distance,mean_signed_token_distance
aime_2025,29.0,511.413793,112.0,236.172414
codeforces,112.0,433.258929,67.0,-316.455357
deepmath_103k,139.0,361.705036,85.0,-170.050360
if_sft_data_verified,305.0,179.895082,21.0,-12.760656
llama_nemotron,127.0,337.062992,47.0,-137.787402
medical_o1,76.0,303.236842,171.5,-115.315789
numinamath_1_5,243.0,647.613169,214.0,-86.600823
ALL domains,1031.0,379.945684,70.0,-100.320078


In [26]:
# (prompt_id, rollout_idx) -> list of verdict dicts (one per backend with an "ok" result), and
# -> stratum string (regardless of whether judging finished yet) -- built once here so Section
# 8's checkbox lookup is O(1) instead of re-filtering the dataframes on every click. Stored
# per dataset (Section 8 picks whichever dataset's lookup to use via its selector).
for tag, d in ds.items():
    llm_judge_results_df = d["llm_judge_results_df"]
    llm_judge_sample_df = d["llm_judge_sample_df"]
    onset_quote_match_df = d.get("onset_quote_match_df", pd.DataFrame())

    # (backend, prompt_id, rollout_idx) -> verbatim-matched onset_quote's token position
    # (Section 8's onset_quote marker toggle), only for onset_quote candidates that were
    # actually found in the rollout's own text (cell above) -- an unmatched/hallucinated
    # onset_quote has no real position to mark, so it's left out of this lookup entirely.
    onset_match_lookup = {}
    if not onset_quote_match_df.empty:
        for row in onset_quote_match_df.itertuples(index=False):
            if row.matched:
                onset_match_lookup[(row.backend, row.prompt_id, int(row.rollout_idx))] = row.llm_onset_position

    llm_judge_lookup = {}
    if not llm_judge_results_df.empty:
        for row in llm_judge_results_df[llm_judge_results_df["status"] == "ok"].itertuples(index=False):
            llm_judge_lookup.setdefault((row.prompt_id, int(row.rollout_idx)), []).append({
                "backend": row.backend,
                "is_degenerating": bool(row.is_degenerating),
                "onset_quote": row.onset_quote,
                "confidence": row.confidence,
                "reasoning": row.reasoning,
                "onset_token_position": onset_match_lookup.get(
                    (row.backend, row.prompt_id, int(row.rollout_idx))
                ),
            })

    llm_judge_stratum_lookup = (
        dict(zip(
            zip(llm_judge_sample_df["prompt_id"], llm_judge_sample_df["rollout_idx"]),
            llm_judge_sample_df["stratum"],
        ))
        if not llm_judge_sample_df.empty else {}
    )
    d["llm_judge_lookup"] = llm_judge_lookup
    d["llm_judge_stratum_lookup"] = llm_judge_stratum_lookup
    print(f"[{tag}] {len(llm_judge_lookup)} rollout(s) with at least one judge verdict, available to Section 8's explorer.")


[apertus-8b-instruct] 1170 rollout(s) with at least one judge verdict, available to Section 8's explorer.


[apertus1p5-capfilter-linear-it8816] 2837 rollout(s) with at least one judge verdict, available to Section 8's explorer.


[apertus1p5-sft256k-4200] 1111 rollout(s) with at least one judge verdict, available to Section 8's explorer.


## 8. Interactive prompt/rollout explorer

Pick a **dataset** first (one of the three builds from Section 1), then a
**domain**, then a **prompt_id** within it, to see per-rollout stats
(how many of its 10 rollouts are repetition-degenerate / have an LRS repeat,
LRS length & period ranges, entropy range, and which stratum, if any,
each rollout was selected into for LLM-judge calibration, per Section 7).
Then, next to **Render generation text**: pick a **rollout_idx**, optionally
cap how many leading tokens to display (**max tokens**, -1 or any invalid
value shows all of them), toggle which signals to overlay, and click render.

Two different visual encodings are used, since the two kinds of signal mean
different things (Section 6):

- **`repetition_score`** / **`entropy`** (per-token, continuous) are drawn as
  thin colored underline segments beneath each generated token (one row per
  active signal, gray = undefined/NaN at that position), with a gradient
  legend at the top.
- **LRS repeat** (one whole-rollout match, not per-token) is drawn instead as
  a colored *background* behind the repeating tokens themselves, spanning the
  full recovered repeat span(s) (`lrs_region_starts`/`lrs_region_ends`, which can
  reach beyond the two occurrences the core algorithm originally found, and can
  be more than one disjoint span if the unit is atomic and repeats with
  unrelated content in between rather than back-to-back), alternating between
  two shades every `lrs_period` tokens within each span, so the stripes directly
  show how many times the unit repeats. A caption above the render states the
  exact numbers/spans and the decoded repeating unit's text. A **LRS field set**
  toggle next to the checkbox switches between Section 6's two field sets --
  `plain` (`lrs_*`) or `digit-run normalized` (`lrs_*_normalized_growing`,
  the default) -- so the same rollout can be re-rendered under either one to
  compare where each locates the repeat.
- **LLM judge verdict** (opt-in checkbox, off by default) appends a new
  section at the very end of the figure with every backend's verdict for
  that rollout: `is_degenerating`, `confidence`, `onset_quote` (a verbatim
  quote marking where degeneration begins, if flagged), and `reasoning`.
  Only available for the rollouts in Section 7's calibration sample;
  anything else shows a note instead of a verdict.
- **onset_quote marker** (opt-in checkbox, off by default) draws a solid
  orange vertical line at the token position where the LLM judge's
  `onset_quote` was found verbatim (Section 7/cell 26). Silently draws
  nothing for a rollout that isn't degenerating, isn't in the calibration
  sample, or whose `onset_quote` didn't match verbatim -- there's no
  position to mark in any of those cases.
- **repetition_score onset marker** (opt-in checkbox, off by default) draws
  a dashed green vertical line at the first token where `repetition_score`
  crosses a threshold you can type in next to the checkbox (default `0.8`).
  This input is independent of the pipeline's actual degeneration
  threshold -- a fixed note next to it states that the real one
  (`label_module.DEFAULT_DEGENERATION_THRESHOLD = 0.8`) never changes;
  moving this input only changes what gets marked in this rendering, so
  you can freely try other cutoffs without confusing them for the real
  one. Silently draws nothing if the rollout never crosses the threshold.

Turning on both onset markers together on the same rollout is the fastest
way to see, visually, whether `onset_quote` and `repetition_score` agree on
*where* degeneration starts, not just *whether* it does.

The figure is shown inline and saved as both `.png` (quick viewing) and
`.pdf` (reports/papers) under
`notebooks/figures/<dataset_tag>/<domain>/<prompt_id>/rollout_<idx>[_<signals>].{png,pdf}`,
where `<dataset_tag>` is the selected dataset (Section 1) and `<signals>`
encodes exactly which checkboxes were on (e.g. `_repetition-lrs-llmjudge`;
omitted entirely if none were checked).

In [27]:
import ipywidgets as widgets
import matplotlib as mpl
from matplotlib.patches import Rectangle

FIGURES_DIR = REPO_ROOT / "notebooks" / "figures"

# --- per-token continuous metrics: color scale + legend text for each signal ---
# LRS is deliberately NOT in here -- it's one whole-rollout match, not a per-token
# array, so it gets its own background-highlight treatment below instead of an
# underline row (see METRIC_SPECS docstring-equivalent note in Section 8's intro cell).
METRIC_SPECS = {
    "repetition_score": dict(
        short="repetition", cmap="Reds", vmin=0.0, vmax=1.0,
        label="repetition_score  (1 - bigram TTR, 256-token window; 0=diverse -> 1=fully repetitive; degeneration threshold=0.8)",
    ),
    "entropy": dict(
        short="entropy", cmap="Blues", vmin=0.0, vmax=None,
        label="entropy  (next-token Shannon entropy, nats i.e. natural-log units, pre-temperature/top-p; darker=more uncertain; scale is per-rendering)",
    ),
}

# Two alternating shades used to highlight the LRS repeat region: one swatch per
# lrs_period-token unit, so consecutive repeats are visually distinguishable.
_LRS_STRIPE_COLORS = [mpl.colormaps["Purples"](0.35), mpl.colormaps["Purples"](0.60)]

# Vertical onset markers (Section 8's two onset-marker toggles) -- a solid orange line
# for the LLM judge's onset_quote position, a dashed green line for where repetition_score
# first crosses a (manually adjustable) threshold. Distinct colors/styles so both can be
# shown at once and compared directly on the same rendered rollout.
ONSET_QUOTE_MARKER_COLOR = "#d95f02"
REPETITION_ONSET_MARKER_COLOR = "#1b9e77"


def _display_token_text(text):
    """Make a decoded token's text safe to lay out as a single monospace cell."""
    text = text.replace("\n", "↵").replace("\r", "↵").replace("\t", "→")
    return text if text else "·"  # zero-width decode (rare) -> visible placeholder


def _wrap_tokens(token_texts, wrap_width):
    """Greedily pack decoded token strings into lines of <= wrap_width columns.

    Returns a list of lines; each line is a list of (token_pos, text, start_col, end_col),
    token_pos indexing into the (possibly truncated) displayed token sequence.
    """
    lines, current, col = [], [], 0
    for pos, text in enumerate(token_texts):
        width = len(text)
        if current and col + width > wrap_width:
            lines.append(current)
            current, col = [], 0
        current.append((pos, text, col, col + width))
        col += width
    lines.append(current)
    return lines


def _token_color(value, cmap, vmin, vmax):
    if value is None or np.isnan(value):
        return (0.55, 0.55, 0.55, 1.0)  # gray: undefined/NaN for this position
    norm = 0.0 if vmax <= vmin else (value - vmin) / (vmax - vmin)
    return mpl.colormaps[cmap](min(max(norm, 0.0), 1.0))


def _measure_char_width_axes_fraction(fig, ax, fontsize):
    """axes-fraction width of one monospace character, measured via the real renderer."""
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    probe = ax.text(0, 0, "M" * 40, family="monospace", fontsize=fontsize, transform=ax.transAxes, alpha=0)
    char_px = probe.get_window_extent(renderer=renderer).width / 40
    probe.remove()
    return char_px / ax.get_window_extent(renderer=renderer).width


def render_generation_figure(
    dataset_tag, domain, prompt_id, rollout_idx,
    show_repetition=True, show_lrs_region=True, show_entropy=True, show_llm_judge=False,
    show_onset_quote_marker=False, show_repetition_onset_marker=False,
    repetition_onset_threshold=0.8,
    max_tokens=-1,
    entropy_vmin=None, entropy_vmax=None,
    wrap_width=110, fontsize=8.5,
    lrs_variant="normalized_growing",
):
    """entropy_vmin/entropy_vmax override METRIC_SPECS["entropy"]'s own vmin/vmax
    for just this call, None meaning 'use the metric's own default' -- vmin=0.0,
    vmax=auto-scaled to this rollout's own max entropy (see below). Lets Section 8's
    entropy min/max inputs fix a shared scale across renders instead of every render
    re-normalizing to its own data, which makes entropy heat hard to compare rollout
    to rollout.

    lrs_variant selects which of the two LRS field sets (Section 6) the background
    highlight/caption reads from: "plain" (`lrs_*`, exact-match only) or
    "normalized_growing" (`lrs_*_normalized_growing`, digit-run-normalized -- the
    onset-position signal used elsewhere in this project, so it's the default here).

    show_onset_quote_marker draws a vertical marker at the LLM judge's onset_quote
    position (only when a verbatim match exists -- see cell 26/28), and
    show_repetition_onset_marker draws one at the first token where repetition_score
    exceeds repetition_onset_threshold (independent of the pipeline's own fixed 0.8
    threshold -- this only changes what gets marked in this rendering). Both are
    silently omitted (no marker, no note) when there's nothing to mark: a
    non-degenerating rollout, an onset_quote that didn't match verbatim, or a
    repetition_score that never crosses the chosen threshold.
    """
    d = ds[dataset_tag]
    prompts_df = d["prompts_df"]
    generations_df = d["generations_df"]
    labels_df = d["labels_df"]
    llm_judge_lookup = d["llm_judge_lookup"]
    tokenizer = get_tokenizer(dataset_tag)

    prompt_row = prompts_df.loc[prompts_df["prompt_id"] == prompt_id].iloc[0]
    gen_row = generations_df.loc[
        (generations_df["prompt_id"] == prompt_id) & (generations_df["rollout_idx"] == rollout_idx)
    ].iloc[0]
    label_match = labels_df.loc[
        (labels_df["prompt_id"] == prompt_id) & (labels_df["rollout_idx"] == rollout_idx)
    ]
    label_row = label_match.iloc[0] if not label_match.empty else None

    token_ids = [int(t) for t in gen_row["generated_token_ids"]]
    num_tokens_total = len(token_ids)

    if max_tokens == -1:
        n_display = num_tokens_total
        max_tokens_note = None
    elif isinstance(max_tokens, (int, np.integer)) and 1 <= max_tokens <= num_tokens_total:
        n_display = max_tokens
        max_tokens_note = None
    else:
        n_display = num_tokens_total
        max_tokens_note = (
            f"max_tokens={max_tokens!r} is invalid (must be -1, or an integer in "
            f"[1, {num_tokens_total}]) -- showing all {num_tokens_total} tokens."
        )

    display_ids = token_ids[:n_display]
    token_texts = [
        _display_token_text(tokenizer.decode([tid], skip_special_tokens=False)) for tid in display_ids
    ]

    active_metrics = [
        key for key, show in (
            ("repetition_score", show_repetition),
            ("entropy", show_entropy),
        )
        if show and label_row is not None
    ]
    metric_arrays, metric_vrange = {}, {}
    for key in active_metrics:
        arr = np.asarray(label_row[key], dtype=float)[:n_display]
        spec = METRIC_SPECS[key]
        if key == "entropy":
            vmin = spec["vmin"] if entropy_vmin is None else entropy_vmin
            vmax = spec["vmax"] if entropy_vmax is None else entropy_vmax
        else:
            vmin = spec["vmin"]
            vmax = spec["vmax"]
        if vmax is None:  # no fixed range given -- scale to what's actually shown
            finite = arr[~np.isnan(arr)]
            vmax = max(float(finite.max()), vmin + 1e-6) if finite.size else vmin + 1e-6
        metric_arrays[key] = arr
        metric_vrange[key] = (vmin, vmax)

    # --- LRS: one whole-rollout match, not a per-token array -- drawn as a background
    # region rather than an underline row. `lrs_region_*` can reach beyond the two
    # occurrences `find_longest_repeated_substring` originally found (see label.py).
    # `lrs_variant` picks which field set to read -- see this function's docstring.
    lrs_field_suffix = "" if lrs_variant == "plain" else "_normalized_growing"
    lrs_variant_label = "plain exact-match" if lrs_variant == "plain" else "digit-run normalized"
    has_lrs_match = label_row is not None and int(label_row[f"lrs_length{lrs_field_suffix}"]) > 0
    draw_lrs = show_lrs_region and has_lrs_match
    if draw_lrs:
        period = int(label_row[f"lrs_period{lrs_field_suffix}"])
        repeat_count = int(label_row[f"lrs_period_repeat_count{lrs_field_suffix}"])
        # Parallel lists, not a single range: the repeating unit can occupy more than
        # one disjoint span (e.g. an atomic phrase repeated twice with unrelated
        # content in between -- see label.py's find_longest_repeated_substring docstring).
        region_starts = [int(s) for s in label_row[f"lrs_region_starts{lrs_field_suffix}"]]
        region_ends = [int(e) for e in label_row[f"lrs_region_ends{lrs_field_suffix}"]]
        unit_ids = [int(t) for t in label_row[f"lrs_unit_token_ids{lrs_field_suffix}"]]
        unit_text = tokenizer.decode(unit_ids, skip_special_tokens=False).replace("\n", "↵").replace("\r", "↵")
        # `lrs_period` for the normalized_growing field set is measured in digit-collapsed
        # (compressed) token units, not real tokens (see label.py's
        # find_longest_repeated_substring_growing_normalized docstring) -- a growing numeral
        # means there's no single real-token period shared by every occurrence. Striping by
        # real token position needs a real-token step size, so use `len(unit_ids)` (the real
        # length of the unit's first occurrence) instead of `period` whenever plotting that
        # field set; for `plain`, `period` already is a real-token count.
        stripe_period = period if lrs_variant == "plain" else len(unit_ids)

    # --- LLM judge verdict(s): looked up by (prompt_id, rollout_idx), not computed here --
    # only rollouts in Section 7's calibration sample have one. Shown as its own section at
    # the very end of the figure (not interleaved with the per-token signals above, since a
    # judge verdict describes the whole rollout, not a position within it).
    # Looked up once regardless of show_llm_judge -- also needed by the onset_quote
    # marker below, which is an independent toggle from the full verdict text.
    judge_entries_all = llm_judge_lookup.get((prompt_id, rollout_idx), [])
    judge_lines = []
    if show_llm_judge:
        if judge_entries_all:
            for entry in judge_entries_all:
                judge_lines.append(
                    f"[{entry['backend']}]  is_degenerating={entry['is_degenerating']}   "
                    f"confidence={entry['confidence']}"
                )
                if entry["onset_quote"]:
                    judge_lines.extend(textwrap.wrap(
                        f"onset_quote: {entry['onset_quote']}",
                        width=wrap_width, subsequent_indent="  ",
                    ))
                judge_lines.extend(textwrap.wrap(
                    f"reasoning: {entry['reasoning']}", width=wrap_width, subsequent_indent="  ",
                ))
                judge_lines.append("")  # blank line between multiple backends' verdicts
            if judge_lines and judge_lines[-1] == "":
                judge_lines.pop()
        else:
            judge_lines = [
                "No LLM-judge verdict available for this rollout (not in the calibration "
                "sample, or not judged yet -- see Section 7)."
            ]

    # --- onset_quote marker: one vertical mark per backend whose onset_quote was
    # verbatim-matched in this rollout's text (cell 26/28) -- a non-degenerating rollout,
    # or a hallucinated/paraphrased onset_quote that didn't match, has no position here
    # and is silently skipped rather than shown as an error.
    onset_quote_marks = []  # list of (token_pos, backend)
    if show_onset_quote_marker:
        for entry in judge_entries_all:
            pos = entry.get("onset_token_position")
            if pos is not None and pos < n_display:
                onset_quote_marks.append((pos, entry["backend"]))

    # --- repetition_score onset marker: first token position where repetition_score
    # exceeds the (manually adjustable) threshold above -- lets the same rollout be
    # compared against where the LLM judge's onset_quote lands. Searched over the FULL
    # array (not the possibly-truncated display) so truncation doesn't change which
    # position counts as "first"; only skipped (not truncated-and-shown) if that
    # position falls beyond what's currently displayed.
    repetition_onset_pos = None
    if show_repetition_onset_marker and label_row is not None:
        rep_arr = np.asarray(label_row["repetition_score"], dtype=float)
        exceed = np.where(rep_arr > repetition_onset_threshold)[0]
        if exceed.size and exceed[0] < n_display:
            repetition_onset_pos = int(exceed[0])

    prompt_lines = textwrap.wrap(prompt_row["prompt_text"], width=wrap_width) or [""]
    token_lines = _wrap_tokens(token_texts, wrap_width)

    header_lines = [
        f"{dataset_tag}  /  {prompt_id}  /  rollout_idx={rollout_idx}  /  domain={domain}",
        f"stop_reason={gen_row['stop_reason']}   num_tokens={int(gen_row['num_tokens'])}"
        + (f"   (showing first {n_display}/{num_tokens_total})" if n_display < num_tokens_total else ""),
    ]
    if max_tokens_note:
        header_lines.append(max_tokens_note)
    if draw_lrs:
        max_region_end = max(region_ends)
        region_note = (
            "" if max_region_end <= n_display
            else f"  (extends to token {max_region_end}, beyond what's displayed)"
        )
        spans_text = ", ".join(f"[{s}:{e})" for s, e in zip(region_starts, region_ends))
        header_line = (
            f"LRS ({lrs_variant_label}): {stripe_period}-token unit repeated {repeat_count}x across "
            f"{len(region_starts)} span(s): {spans_text}{region_note}"
        )
        # Header lines are drawn unwrapped (one advance() per entry, matching
        # n_plain_lines) -- with many spans this can run far past the axes'
        # right edge, which makes plt.tight_layout() shrink the axes to fit
        # it. Since char_w was already measured against the PRE-shrink axes
        # size, that silently desyncs every later token's column position
        # from its actual pixel width, compounding into visible overlap by
        # the end of each line. Capping the line's length keeps it (and
        # everything else) inside the axes so tight_layout never has a
        # reason to touch the axes size.
        header_lines.append(textwrap.shorten(header_line, width=wrap_width, placeholder=" ..."))
        header_lines.append(f'  unit: "{textwrap.shorten(unit_text, width=wrap_width - 9, placeholder=" ...")}"')
    elif show_lrs_region and label_row is not None:
        header_lines.append(
            f"LRS ({lrs_variant_label}): no repeat found "
            f"(below the {label_module.DEFAULT_LRS_MIN_LENGTH}-token minimum)"
        )
    if show_onset_quote_marker and onset_quote_marks:
        marks_text = ", ".join(f"{backend} @ token {pos}" for pos, backend in onset_quote_marks)
        header_lines.append(f"onset_quote marker (orange): {marks_text}")
    if show_repetition_onset_marker and repetition_onset_pos is not None:
        header_lines.append(
            f"repetition_score onset marker (green, dashed, threshold={repetition_onset_threshold}): "
            f"token {repetition_onset_pos}"
        )

    # --- figure sizing: sum up vertical "row units" up front, matching the draw pass exactly ---
    LINE_UNIT = 1.0                  # one plain text line (header/prompt/section-label)
    TEXT_TO_SEG_GAP = 0.85           # generated-text line -> its first metric underline row
                                      # (needs to clear the text's own descenders -- anything
                                      # much smaller visibly touches the text)
    SEG_UNIT = 0.34                  # one metric underline row -> the next one (thin bar to
                                      # thin bar, no descenders involved, so these stack tightly)
    NEXT_LINE_GAP_REDUCTION = 0.35   # a generated-text line that follows another one's metric
                                      # underlines doesn't need a FULL LINE_UNIT of headroom --
                                      # nothing above it has descenders to clear -- so shrink
                                      # that specific transition to tighten the visible gap
                                      # between the last underline row and the next line of text
    LEGEND_UNIT = 2.0                # one metric's legend entry: a full label line + a full bar line

    n_metrics = len(active_metrics)
    # LRS gets one more legend entry (label + swatch row), same LEGEND_UNIT budget as a
    # continuous metric, even though it draws no extra per-line row (see below) -- it's
    # only ever drawn as a background behind the SAME text row.
    n_legend_entries = n_metrics + (1 if draw_lrs else 0)
    legend_units = (n_legend_entries * LEGEND_UNIT + 1.0) if (n_metrics or draw_lrs) else 0.0
    n_plain_lines = len(header_lines) + 1 + 1 + len(prompt_lines) + 1 + 1  # + 2 blanks + 2 section labels
    if show_llm_judge:
        n_plain_lines += 1 + 1 + len(judge_lines)  # + 1 blank + 1 section label + judge lines
    seg_rows_unit = (TEXT_TO_SEG_GAP + max(0, n_metrics - 1) * SEG_UNIT) if n_metrics else 0.0

    n_token_lines = len(token_lines)
    if n_token_lines:
        first_line_unit = LINE_UNIT + seg_rows_unit
        later_text_unit = (LINE_UNIT - NEXT_LINE_GAP_REDUCTION) if n_metrics else LINE_UNIT
        later_line_unit = later_text_unit + seg_rows_unit
        token_lines_total_unit = first_line_unit + max(0, n_token_lines - 1) * later_line_unit
    else:
        token_lines_total_unit = 0.0
    total_units = legend_units + n_plain_lines * LINE_UNIT + token_lines_total_unit

    line_height = 0.16  # inches per unit, monospace font
    fig_height = max(2.0, total_units * line_height + 0.6)
    fig_width = 10.0

    # Prompt/generated text often contains literal "$...$" (e.g. AIME problem
    # statements) -- disable mathtext parsing so it renders as plain text
    # instead of matplotlib trying (and failing) to parse it as LaTeX math.
    with plt.rc_context({"text.parse_math": False}):
        fig, ax = plt.subplots(figsize=(fig_width, fig_height))
        ax.axis("off")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)

        char_w = _measure_char_width_axes_fraction(fig, ax, fontsize)
        dy = 1.0 / (total_units + 2)
        y = [1.0]  # mutable cursor (top edge of the next row), shared by nested helpers

        def advance(units):
            y[0] -= units * dy
            return y[0]

        def draw_text(text, bold=False, size=fontsize, x=0.01, ha="left", style="normal"):
            ax.text(
                x, y[0], text, family="monospace", fontsize=size, weight="bold" if bold else "normal",
                style=style, ha=ha, va="top", transform=ax.transAxes,
            )

        # --- legend: one entry per active metric (gradient bar), plus one for LRS
        # (alternating-shade swatch) if active. Each entry is TWO full LINE_UNIT-equivalent
        # rows (label, then bar/swatch) so it clears the label's own glyph height -- a
        # half-row gap here visually overlaps the label's descenders instead of sitting
        # cleanly below it.
        if active_metrics or draw_lrs:
            for key in active_metrics:
                spec = METRIC_SPECS[key]
                vmin, vmax = metric_vrange[key]
                advance(1.0)
                draw_text(spec["label"], bold=True)
                bar_top = advance(1.0)
                bar_x0, bar_x1, bar_h = 0.01, 0.35, dy * 0.35
                ax.imshow(
                    np.linspace(0, 1, 256).reshape(1, -1),
                    extent=(bar_x0, bar_x1, bar_top - bar_h, bar_top),
                    transform=ax.transAxes, aspect="auto", cmap=spec["cmap"], vmin=0, vmax=1,
                )
                ax.text(bar_x0 - 0.008, bar_top - bar_h / 2, f"{vmin:.2g}", family="monospace",
                         fontsize=fontsize - 1.5, va="center", ha="right", transform=ax.transAxes)
                ax.text(bar_x1 + 0.008, bar_top - bar_h / 2, f"{vmax:.2g}", family="monospace",
                         fontsize=fontsize - 1.5, va="center", ha="left", transform=ax.transAxes)
            if draw_lrs:
                advance(1.0)
                draw_text(
                    f"LRS repeat region -- {lrs_variant_label}  (alternating shades = consecutive "
                    "repeated token-units; see caption above for exact counts)",
                    bold=True,
                )
                swatch_top = advance(1.0)
                swatch_h = dy * 0.35
                n_swatches = 8
                swatch_x0, swatch_x1 = 0.01, 0.35
                swatch_w = (swatch_x1 - swatch_x0) / n_swatches
                for si in range(n_swatches):
                    color = _LRS_STRIPE_COLORS[si % 2]
                    x0 = swatch_x0 + si * swatch_w
                    ax.add_patch(Rectangle(
                        (x0, swatch_top - swatch_h), swatch_w * 0.92, swatch_h,
                        transform=ax.transAxes, facecolor=color, edgecolor="none",
                    ))
            if active_metrics:
                advance(0.5)
                draw_text(
                    "gray segment beneath a token = value undefined (NaN) at that position",
                    style="italic", size=fontsize - 1.5,
                )
            advance(0.5)

        for line in header_lines:
            advance(LINE_UNIT)
            draw_text(line, bold=True)
        advance(LINE_UNIT)  # blank
        advance(LINE_UNIT)
        draw_text("PROMPT:", bold=True)
        for line in prompt_lines:
            advance(LINE_UNIT)
            draw_text(line)
        advance(LINE_UNIT)  # blank
        advance(LINE_UNIT)
        draw_text("GENERATED TEXT:", bold=True)

        # Each generated-text line's own text needs a full LINE_UNIT of headroom only
        # the first time (it follows the "GENERATED TEXT:" label, ordinary text-to-text
        # spacing) -- every later line follows the PREVIOUS line's metric underlines
        # instead, which have no descenders to clear, so that transition is shrunk by
        # NEXT_LINE_GAP_REDUCTION to keep the underlines snug against both neighbors.
        for i, line in enumerate(token_lines):
            text_unit = LINE_UNIT if (i == 0 or not active_metrics) else LINE_UNIT - NEXT_LINE_GAP_REDUCTION
            text_y = advance(text_unit)
            for pos, text, start_col, end_col in line:
                # LRS region highlight: a background behind the token, drawn in the SAME
                # row as the text itself (not an extra row like the underline metrics
                # below) -- alternates shade every `stripe_period` tokens from region_start
                # (real-token step size -- see `stripe_period`'s definition above for why
                # this isn't simply `period` for the normalized_growing field set).
                if draw_lrs:
                    for r_start, r_end in zip(region_starts, region_ends):
                        if r_start <= pos < r_end:
                            stripe_idx = (pos - r_start) // stripe_period
                            color = _LRS_STRIPE_COLORS[stripe_idx % 2]
                            x0, x1 = 0.01 + start_col * char_w, 0.01 + end_col * char_w
                            hl_h = dy * LINE_UNIT * 0.85
                            ax.add_patch(Rectangle(
                                (x0, text_y - hl_h), max(x1 - x0, char_w * 0.05), hl_h,
                                transform=ax.transAxes, facecolor=color, edgecolor="none", zorder=1,
                            ))
                            break
                ax.text(0.01 + start_col * char_w, text_y, text, family="monospace", fontsize=fontsize,
                         va="top", transform=ax.transAxes, zorder=2)
            # All active metrics' underlines cluster tightly right below the text: only
            # the first one needs the bigger TEXT_TO_SEG_GAP to clear the text itself,
            # the rest stack with the much tighter SEG_UNIT (bar-to-bar, no glyphs).
            for m_idx, key in enumerate(active_metrics):
                spec = METRIC_SPECS[key]
                vmin, vmax = metric_vrange[key]
                arr = metric_arrays[key]
                seg_top = advance(TEXT_TO_SEG_GAP if m_idx == 0 else SEG_UNIT)
                seg_h = dy * SEG_UNIT * 0.6
                for pos, _text, start_col, end_col in line:
                    color = _token_color(arr[pos], spec["cmap"], vmin, vmax)
                    x0, x1 = 0.01 + start_col * char_w, 0.01 + end_col * char_w
                    pad = min(0.15 * char_w, (x1 - x0) * 0.2)
                    ax.add_patch(Rectangle(
                        (x0 + pad, seg_top - seg_h), max(x1 - x0 - 2 * pad, char_w * 0.05), seg_h,
                        transform=ax.transAxes, facecolor=color, edgecolor="none",
                    ))

            # --- onset markers: a thin vertical line spanning from this line's text
            # down through its own metric underlines (or a fixed fallback height if no
            # metric is active, so the marker is still visible on its own). Two markers
            # can land on the same token -- nudged apart in x so both stay visible
            # instead of overdrawing each other.
            if show_onset_quote_marker or show_repetition_onset_marker:
                marker_top = text_y
                marker_bottom = y[0] if active_metrics else text_y - dy * LINE_UNIT * 0.7
                nudge = char_w * 0.12
                for pos, _text, start_col, _end_col in line:
                    x = 0.01 + start_col * char_w
                    if show_onset_quote_marker and any(p == pos for p, _b in onset_quote_marks):
                        ax.plot(
                            [x - nudge, x - nudge], [marker_bottom, marker_top],
                            color=ONSET_QUOTE_MARKER_COLOR, linewidth=2.2, zorder=4,
                            solid_capstyle="butt",
                        )
                    if show_repetition_onset_marker and repetition_onset_pos == pos:
                        ax.plot(
                            [x + nudge, x + nudge], [marker_bottom, marker_top],
                            color=REPETITION_ONSET_MARKER_COLOR, linewidth=2.2, zorder=4,
                            linestyle="--",
                        )

        # --- LLM judge verdict section: a plain-text block at the very end, after all
        # generated-text lines -- a verdict describes the whole rollout, so it doesn't
        # belong interleaved with the per-token rows above.
        if show_llm_judge:
            advance(LINE_UNIT)  # blank
            advance(LINE_UNIT)
            draw_text("LLM JUDGE VERDICT:", bold=True)
            for line in judge_lines:
                advance(LINE_UNIT)
                draw_text(line)

        plt.tight_layout()

    metric_shorts = [METRIC_SPECS[k]["short"] for k in active_metrics]
    if draw_lrs:
        metric_shorts.append("lrs" if lrs_variant == "plain" else "lrs-norm")
    if show_llm_judge:
        metric_shorts.append("llmjudge")
    if show_onset_quote_marker:
        metric_shorts.append("onsetquote")
    if show_repetition_onset_marker:
        metric_shorts.append("reponset")
    suffix = ("_" + "-".join(metric_shorts)) if metric_shorts else ""
    out_dir = FIGURES_DIR / dataset_tag / domain / prompt_id
    out_dir.mkdir(parents=True, exist_ok=True)
    png_path = out_dir / f"rollout_{rollout_idx}{suffix}.png"
    pdf_path = out_dir / f"rollout_{rollout_idx}{suffix}.pdf"
    fig.savefig(png_path, dpi=150, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    return fig, png_path, pdf_path

In [28]:
# Defensive against re-running this cell without a kernel restart: ipywidgets'
# .observe()/.on_click() ADD handlers rather than replacing them, so re-executing
# this cell twice (e.g. while debugging an earlier cell and re-running downstream
# ones) would fire every callback twice per click. Closing any widgets left over
# from a previous run of THIS cell disables their old comms first, so a stray
# double-fire can't happen even if an old widget is still visible somewhere.
for _name in (
    "dataset_dd", "domain_dd", "prompt_dd", "rollout_dd", "max_tokens_input",
    "cb_repetition", "cb_lrs", "lrs_variant_tb", "cb_entropy", "entropy_vmin_input", "entropy_vmax_input",
    "cb_llm_judge", "cb_onset_quote_marker", "cb_repetition_onset", "repetition_onset_threshold_input",
    "repetition_onset_note", "render_btn", "stats_out", "render_out",
):
    _old = globals().get(_name)
    if _old is not None and hasattr(_old, "close"):
        _old.close()

from IPython.display import HTML, FileLink

# A dropdown with no real options yet (nothing to pick until you've picked the
# level above it, or the selected dataset has nothing generated yet) is set
# .disabled below rather than just left showing an empty list -- ipywidgets'
# own disabled look is a subtle opacity change that's easy to miss, so this adds
# a clearly-different gray fill/text and drops the native select arrow (which
# otherwise still invites a click even though there's nothing to choose).
# Scoped to `.degen-explorer` (added to the outermost VBox below) so it doesn't
# leak into other widgets elsewhere in the notebook. The var(--jp-...) values
# are JupyterLab theme variables (fall back to the literal color outside Lab),
# so the disabled look stays legible in both light and dark theme.
display(HTML(
    """<style>
    .degen-explorer select:disabled {
        background-color: var(--jp-layout-color2, #e0e0e0) !important;
        color: var(--jp-ui-font-color2, #888888) !important;
        opacity: 1 !important;
        -webkit-appearance: none !important;
        -moz-appearance: none !important;
        appearance: none !important;
        background-image: none !important;
        cursor: not-allowed;
    }
    </style>"""
))

dataset_dd = widgets.Dropdown(options=DATASET_TAGS, description="dataset:")
domain_dd = widgets.Dropdown(options=[], description="domain:")
prompt_dd = widgets.Dropdown(options=[], description="prompt_id:")

# rollout_idx + the render controls all live together, since they're only
# needed for "Render generation text" (unlike dataset/domain/prompt_id, which
# also drive the stats table above).
rollout_dd = widgets.Dropdown(options=[], description="rollout_idx:")
max_tokens_input = widgets.IntText(
    value=-1, description="max tokens:", layout=widgets.Layout(width="150px"),
    tooltip="Number of leading generated tokens to display. -1 (default) or any invalid value shows all tokens.",
)
cb_repetition = widgets.Checkbox(value=True, description="repetition_score", indent=False)
cb_lrs = widgets.Checkbox(value=True, description="LRS repeat (highlight region)", indent=False)
# Which of Section 6's two LRS field sets to highlight/caption -- "plain" (`lrs_*`,
# exact-match only) or "normalized_growing" (`lrs_*_normalized_growing`, digit-run
# normalized -- the onset-position signal used elsewhere in this project, hence the
# default here). Lets the same rollout be re-rendered under either variant to compare.
lrs_variant_tb = widgets.ToggleButtons(
    options=[("plain", "plain"), ("digit-run normalized", "normalized_growing")],
    value="normalized_growing",
    description="LRS field set:",
    style={"description_width": "auto"},
    tooltip="Which LRS field set the highlight/caption reads from -- see Section 6.",
)
cb_entropy = widgets.Checkbox(value=True, description="entropy", indent=False)
# Fixes the entropy color scale across renders instead of every render
# auto-normalizing to its own max, which otherwise makes entropy heat hard to
# compare rollout to rollout. Negative (default: min=0, max=-1) means "use the
# metric's own default for that bound" -- vmin=0.0, vmax=auto-scaled per-render.
# The description label ("min:"/"max:") eats space from `layout.width` unless
# description_width is fixed separately -- without it, the actual number field
# left over is too narrow to show its own value.
_vrange_style = {"description_width": "35px"}
entropy_vmin_input = widgets.FloatText(
    value=0.0, description="min:", style=_vrange_style, layout=widgets.Layout(width="150px"),
    tooltip="Entropy color-scale lower bound. Negative = use the default (0).",
)
entropy_vmax_input = widgets.FloatText(
    value=-1.0, description="max:", style=_vrange_style, layout=widgets.Layout(width="150px"),
    tooltip="Entropy color-scale upper bound. Negative (default -1) = auto-scale "
    "to this rollout's own max entropy.",
)
# Off by default: most rollouts aren't in Section 7's calibration sample, so an
# always-on checkbox would show a "no verdict" note on nearly every render.
cb_llm_judge = widgets.Checkbox(value=False, description="LLM judge verdict", indent=False)

# Two independent onset markers (off by default, same reasoning as cb_llm_judge above --
# most rollouts have nothing to mark either way): a vertical line at the LLM judge's
# onset_quote position, and one at the first token where repetition_score crosses a
# threshold. Letting both be toggled on together is the point -- it's the fastest way to
# see, on one rendered rollout, whether the two signals agree on where degeneration starts.
cb_onset_quote_marker = widgets.Checkbox(
    value=False, description="onset_quote marker (orange)", indent=False,
    tooltip="Vertical marker at the LLM judge's onset_quote position, if it was found "
    "verbatim in this rollout's text (Section 7). No-op if not degenerating / no match.",
)
cb_repetition_onset = widgets.Checkbox(
    value=False, description="repetition_score onset marker (green, dashed)", indent=False,
    tooltip="Vertical marker at the first token where repetition_score exceeds the "
    "threshold below. No-op if it never crosses that threshold.",
)
# Deliberately independent from label_module.DEFAULT_DEGENERATION_THRESHOLD -- this only
# changes what gets MARKED in this rendering, not any actual label/metric computed by the
# pipeline (see the note displayed next to it).
repetition_onset_threshold_input = widgets.FloatText(
    value=label_module.DEFAULT_DEGENERATION_THRESHOLD, description="threshold:",
    style={"description_width": "60px"}, layout=widgets.Layout(width="160px"),
    tooltip="Manually adjustable threshold for the repetition_score onset marker only.",
)
repetition_onset_note = widgets.HTML(
    "<i>note: the pipeline's actual degeneration threshold is fixed at "
    f"{label_module.DEFAULT_DEGENERATION_THRESHOLD}</i>"
)
render_btn = widgets.Button(description="Render generation text", button_style="primary")

stats_out = widgets.Output()
render_out = widgets.Output()

THRESHOLD = label_module.DEFAULT_DEGENERATION_THRESHOLD
RENDER_BTN_LABEL = render_btn.description


def _select_first_if_unset(dd):
    """Assigning `.options` on a freshly-constructed (options=[]) dropdown does
    NOT auto-select the first entry the way constructing it with non-empty
    options upfront would (an ipywidgets quirk -- `.value` silently stays
    `None`), which would otherwise leave every downstream dropdown looking
    populated but unselected on first load, or after switching datasets, until
    the user manually re-clicks the same option. Only touches `.value` when
    it's still unset -- if the dropdown already has a value that's still valid
    for the new options (e.g. staying on the same domain name across a dataset
    switch), that selection is left alone rather than jumping back to the
    first option.
    """
    if dd.value is None and dd.options:
        first = dd.options[0]
        dd.value = first[1] if isinstance(first, tuple) else first


# ipywidgets can re-fire a "value" observer even when the value hasn't logically
# changed -- e.g. when the front-end resyncs its widget state back to the kernel
# on re-render, the incoming-state handler doesn't always do the same equality
# check a normal Python trait assignment would. Observed directly: selecting one
# prompt_id could print its stats table anywhere from 2 to 8+ times. Tracking the
# last-rendered selection and skipping a redundant re-fire is a standard guard
# against exactly this class of bug (the same pattern used in React/Vue/Qt-signal
# code to no-op a handler invoked more often than the UI state actually changed).
# Keyed by (dataset_tag, domain[, prompt_id]) rather than just domain/prompt_id --
# the same domain name and prompt_id exist in all three datasets (same prompt
# sample), so the guard must include which dataset is selected. Below, each
# level unconditionally calls the next level once it's done (not just from its
# own widget's "value" observer) specifically so that a dataset switch which
# happens to leave domain_dd/prompt_dd's *string* value unchanged (e.g. same
# domain name, or even the same prompt_id, valid in more than one dataset)
# still re-derives everything beneath it for the newly-selected dataset --
# relying on "value" alone changing would otherwise silently skip that refresh.
#
# The clears below are deliberately wait=False (immediate): the guard cannot catch
# *every* redundant fire (successive fires can carry genuinely different transient
# .value states as the front-end resyncs), and clear_output(wait=True) defers the
# clear until the next output arrives, which fails to collapse those rapid back-to-
# back writes -- they accumulate as N stacked copies of the stats block instead.
# An immediate clear wipes the previous block before each write, so only the final
# (correct) render survives no matter how many times the observer fires.
_last_rendered_domain_key = {"value": None}
_last_rendered_prompt_key = {"value": None}


def _prompt_option_label(tag, prompt_id, sub):
    llm_judge_lookup = ds[tag]["llm_judge_lookup"]
    n_capped = int((sub["stop_reason"] == "length").sum())
    # Surfaced here (not just on the rollout_idx dropdown) so a prompt with zero judged
    # rollouts -- most of them, only a small fraction of rollouts total are judged -- is
    # obvious before you even open it, instead of finding out after picking it.
    n_judged = sum(1 for r in sub["rollout_idx"] if (prompt_id, int(r)) in llm_judge_lookup)
    return (
        f"{prompt_id}  ({n_capped}/{len(sub)} hit max tokens limit, "
        f"{n_judged}/{len(sub)} llm-judged)"
    )


def update_domains(*_):
    tag = dataset_dd.value
    rollout_signal_df = ds[tag]["rollout_signal_df"]
    domain_options = sorted(rollout_signal_df["domain"].unique()) if not rollout_signal_df.empty else []
    domain_dd.options = domain_options
    # Nothing generated/labeled for this dataset yet -- gray the selector out
    # instead of leaving it looking pickable with an empty list (see the CSS
    # injected above).
    domain_dd.disabled = not domain_options
    _select_first_if_unset(domain_dd)
    update_prompts()  # always cascade -- see the note above on why this can't
                       # just rely on domain_dd's own "value" observer firing


def update_prompts(*_):
    tag = dataset_dd.value
    domain = domain_dd.value
    key = (tag, domain)
    if key != _last_rendered_domain_key["value"]:
        _last_rendered_domain_key["value"] = key
        rollout_signal_df = ds[tag]["rollout_signal_df"]
        if domain is None or rollout_signal_df.empty:
            prompt_dd.options = []
            prompt_dd.disabled = True
        else:
            sub = rollout_signal_df[rollout_signal_df["domain"] == domain]
            prompt_ids = sorted(sub["prompt_id"].unique())
            prompt_dd.options = [
                (_prompt_option_label(tag, pid, sub[sub["prompt_id"] == pid]), pid) for pid in prompt_ids
            ]
            # A domain must be picked (and have prompts) before prompt_id means
            # anything -- grayed out until then.
            prompt_dd.disabled = not prompt_ids
            _select_first_if_unset(prompt_dd)
    update_stats()  # always cascade, same reasoning as update_domains above --
                     # cheap even when guarded above, since update_stats has its
                     # own (tag, domain, prompt_id) guard


def update_stats(*_):
    tag = dataset_dd.value
    domain = domain_dd.value
    prompt_id = prompt_dd.value
    key = (tag, domain, prompt_id)
    if key == _last_rendered_prompt_key["value"]:
        return
    _last_rendered_prompt_key["value"] = key
    stats_out.clear_output()
    render_out.clear_output()
    if prompt_id is None:
        rollout_dd.options = []
        rollout_dd.disabled = True
        return
    d = ds[tag]
    rollout_signal_df = d["rollout_signal_df"]
    llm_judge_lookup = d["llm_judge_lookup"]
    sub = rollout_signal_df[rollout_signal_df["prompt_id"] == prompt_id].sort_values("rollout_idx")
    # Label each rollout_idx with whether it has an LLM-judge verdict (Section 7),
    # so you can pick a judged rollout without guessing then checking the checkbox.
    rollout_dd.options = [
        (f"{r}  [llm-judged]" if (prompt_id, int(r)) in llm_judge_lookup else str(r), r)
        for r in sub["rollout_idx"].tolist()
    ]
    # A prompt_id must be picked before rollout_idx means anything -- grayed out
    # until then (mirrors domain_dd/prompt_dd above).
    rollout_dd.disabled = not rollout_dd.options
    _select_first_if_unset(rollout_dd)
    with stats_out:
        n = len(sub)
        n_repetition_degen = int((sub["max_repetition_score"] > THRESHOLD).sum())
        matched = sub[sub["has_lrs_match"]]
        n_lrs_match = len(matched)
        n_llm_judged = int(sub["rollout_idx"].apply(lambda r: (prompt_id, int(r)) in llm_judge_lookup).sum())
        print(f"{prompt_id}  (dataset={tag}, domain={domain}, {n} rollouts)")
        print(f"  repetition-degenerate (max_repetition_score > {THRESHOLD}): {n_repetition_degen}/{n}")
        print(f"  LRS repeat found (>= {label_module.DEFAULT_LRS_MIN_LENGTH} tokens): {n_lrs_match}/{n}")
        if not matched.empty:
            print(f"    lrs_length range: {int(matched['lrs_length'].min())} - {int(matched['lrs_length'].max())} tokens")
            print(f"    lrs_period range: {int(matched['lrs_period'].min())} - {int(matched['lrs_period'].max())} tokens")
        print(f"  mean_entropy range         : {sub['mean_entropy'].min():.3f} - {sub['mean_entropy'].max():.3f}")
        print(f"  LLM-judge verdict available (Section 7): {n_llm_judged}/{n}")
        print()
        display_df = sub[[
            "rollout_idx", "num_tokens", "stop_reason",
            "max_repetition_score", "has_lrs_match", "lrs_length", "lrs_period", "mean_entropy",
        ]].assign(
            repetition_degenerate=lambda dfr: dfr["max_repetition_score"] > THRESHOLD,
            llm_judge_stratum=lambda dfr: [
                d["llm_judge_stratum_lookup"].get((prompt_id, int(r)), None) for r in dfr["rollout_idx"]
            ],
        ).set_index("rollout_idx")
        # Wrapped in a horizontally-scrollable div rather than a bare display(): this
        # table has 9 columns and easily overflows a normal notebook width, which
        # otherwise stretches the whole page instead of just scrolling the table.
        display(HTML(
            '<div style="overflow-x:auto; max-width:100%;">'
            + display_df.to_html()
            + "</div>"
        ))


def on_render_click(_):
    render_out.clear_output(wait=True)
    tag = dataset_dd.value
    domain = domain_dd.value
    prompt_id = prompt_dd.value
    rollout_idx = rollout_dd.value
    if rollout_idx is None:
        with render_out:
            print("Select a rollout_idx first.")
        return

    entropy_vmin = None if entropy_vmin_input.value < 0 else float(entropy_vmin_input.value)
    entropy_vmax = None if entropy_vmax_input.value < 0 else float(entropy_vmax_input.value)

    # Rendering (esp. all signals on a long, untruncated rollout, or the first
    # render for a dataset whose tokenizer hasn't been loaded yet) can take up to
    # ~20s -- disable the button and show a spinner so it doesn't look stuck.
    render_btn.disabled = True
    render_btn.icon = "spinner"
    render_btn.description = "Rendering..."
    with render_out:
        print("Rendering...")
    try:
        fig, png_path, pdf_path = render_generation_figure(
            tag, domain, prompt_id, rollout_idx,
            show_repetition=cb_repetition.value,
            show_lrs_region=cb_lrs.value,
            show_entropy=cb_entropy.value,
            show_llm_judge=cb_llm_judge.value,
            show_onset_quote_marker=cb_onset_quote_marker.value,
            show_repetition_onset_marker=cb_repetition_onset.value,
            repetition_onset_threshold=float(repetition_onset_threshold_input.value),
            max_tokens=max_tokens_input.value,
            entropy_vmin=entropy_vmin,
            entropy_vmax=entropy_vmax,
            lrs_variant=lrs_variant_tb.value,
        )
        render_out.clear_output(wait=True)
        with render_out:
            plt.show()
            # FileLink renders a clickable link (relative to the kernel's cwd, which
            # Jupyter resolves against the notebook's own URL) instead of inert text --
            # click it to open the file directly rather than copy-pasting the path.
            png_rel = os.path.relpath(png_path, start=Path.cwd())
            pdf_rel = os.path.relpath(pdf_path, start=Path.cwd())
            display(FileLink(png_rel, result_html_prefix="Saved PNG: "))
            display(FileLink(pdf_rel, result_html_prefix="Saved PDF: "))
        plt.close(fig)
    finally:
        render_btn.disabled = False
        render_btn.icon = ""
        render_btn.description = RENDER_BTN_LABEL


dataset_dd.observe(update_domains, names="value")
domain_dd.observe(update_prompts, names="value")
prompt_dd.observe(update_stats, names="value")
render_btn.on_click(on_render_click)

# Cascades all the way down to rollout_dd on its own (see the notes on
# update_domains/update_prompts above) -- no separate manual follow-up call
# needed here.
update_domains()

# rollout_idx / max_tokens / the render button share one row; the checkboxes
# stack in their own single column below it (rather than all seven controls
# crammed into one HBox) so they're never at risk of running off the right
# edge and needing a horizontal scroll to reach the button.
controls_row = widgets.VBox([
    widgets.HBox([rollout_dd, max_tokens_input, render_btn]),
    widgets.VBox([
            cb_repetition,
            widgets.HBox([cb_lrs, lrs_variant_tb]),
            widgets.HBox([cb_entropy, entropy_vmin_input, entropy_vmax_input]),
            cb_llm_judge,
            cb_onset_quote_marker,
            widgets.HBox([cb_repetition_onset, repetition_onset_threshold_input, repetition_onset_note]),
        ]),
])

explorer_box = widgets.VBox([
    widgets.HBox([dataset_dd, domain_dd, prompt_dd]),
    stats_out,
    controls_row,
    render_out,
])
explorer_box.add_class("degen-explorer")
display(explorer_box)